# Dissertation figure system — structural redesign

This reporting notebook replaces the earlier ten-figure layout with a fifteen-figure narrative. Every visual answers a distinct methodological or substantive question. The notebook reads only canonical frozen samples, fold assignments, out-of-fold predictions, residuals and robustness summaries. It does not fit, tune, select or retrain any model.

Outputs are written to `Outputs/final_pipeline/dissertation_package_structural`. Each figure is exported as a 400-dpi PNG and editable PDF/SVG, with plotted data, captions, a figure-plan table, a detailed design specification and a scientific QA table.


In [ ]:
# Mount Drive and import the frozen project configuration.
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/GEOG0105')
FINAL_CODE_DIR = PROJECT_DIR / 'CODE' / 'FINAL_PIPELINE'
if str(FINAL_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(FINAL_CODE_DIR))

import config as cfg
print('Project:', cfg.BASE_DIR)


## Reporting environment and visual grammar

Colour is reserved for substantive distinctions: controls, representations, combinations, validation designs and model classes. Fold assignments use a colour-blind-safe categorical palette. Maps retain geographic context; prediction and robustness figures deliberately use different chart types so that scientific questions are visually distinguishable.


In [ ]:
from pathlib import Path
from io import BytesIO
import ast, json, math, re, sys, warnings, zipfile

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patheffects as pe
from matplotlib.colors import TwoSlopeNorm, ListedColormap, BoundaryNorm, Normalize
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns
from scipy.ndimage import gaussian_filter
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.neighbors import BallTree
from PIL import Image, ImageOps

try:
    import geopandas as gpd
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'geopandas', 'pyogrio'])
    import geopandas as gpd

try:
    import contextily as ctx
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'contextily'])
    import contextily as ctx

BUILD_MAIN_FIGURES = True
BUILD_APPENDIX_FIGURES = True
BUILD_TABLES = True
USE_BASEMAP = True

REPORT_DIR = cfg.FINAL_OUTPUT_DIR / 'dissertation_package_structural'
FIGURE_DIR = REPORT_DIR / 'figures'
TABLE_DIR = REPORT_DIR / 'tables'
APPENDIX_DIR = REPORT_DIR / 'appendix'
MANIFEST_DIR = REPORT_DIR / 'manifests'
for p in [REPORT_DIR, FIGURE_DIR, TABLE_DIR, APPENDIX_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

COLORS = {
    'ink': '#222629', 'mid': '#6F767B', 'light': '#D8DCDE', 'paper': '#F7F6F2',
    'control': '#646A70', 'individual': '#2E627D', 'fusion': '#A4652A',
    'random': '#8B9DA8', 'borough': '#2E627D', 'geometric': '#A4652A',
    'ridge': '#74797D', 'xgboost': '#26736A', 'gat': '#8E5B89', 'mlp': '#9A9FA3',
    'negative': '#9B4D4A', 'positive': '#2F756C',
}
FOLD_COLORS = ['#332288', '#88CCEE', '#44AA99', '#DDCC77', '#CC6677']

MODEL_LABELS = {
    'PTAL_spatial_baseline': 'Spatial controls',
    'EPC_controls_sparse': 'Compact property controls',
    'EPC_controls_extensive': 'Privileged extended property controls',
    'SatCLIP': 'Location embedding (SatCLIP)',
    'TESSERA': 'Satellite time-series embedding (TESSERA)',
    'AlphaEarth': 'Annual EO embedding (AlphaEarth)',
    'DINOv2': 'Aerial-image embedding (DINOv2)',
    'StreetView_CLIP_only': 'Street-view embedding (CLIP)',
    'StreetView_CLIP_plus_metadata': 'Street-view embedding (CLIP) + coverage',
    'StreetView_metadata_only': 'Street-view coverage only',
    'Sky_and_space': 'Aerial + satellite information',
    'Street_and_sky': 'Street view + aerial information',
    'Location_and_EO': 'Location + satellite information',
    'All_representations_without_SV_metadata': 'All embeddings',
    'All_representations_plus_SV_metadata': 'Full combination',
}

def short_model(model_id):
    if model_id in MODEL_LABELS:
        return MODEL_LABELS[model_id]
    if '__plus__' in str(model_id):
        base, added = str(model_id).split('__plus__', 1)
        return f'{MODEL_LABELS.get(base, base.replace("_", " "))} + {MODEL_LABELS.get(added, added.replace("_", " "))}'
    return str(model_id).replace('_', ' ')

mpl.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 400, 'font.family': 'DejaVu Sans',
    'font.size': 9.5, 'axes.titlesize': 10.5, 'axes.titleweight': 'semibold',
    'axes.labelsize': 9.5, 'legend.fontsize': 8, 'xtick.labelsize': 8.3,
    'ytick.labelsize': 8.3, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#8E9498', 'axes.linewidth': .7, 'axes.grid': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'text.color': COLORS['ink'],
    'axes.labelcolor': COLORS['ink'], 'xtick.color': '#4D5458', 'ytick.color': '#4D5458',
})
sns.set_theme(style='ticks', context='paper', rc=mpl.rcParams)

figure_records = []

def save_figure(fig, figure_id, title, chapter, source_files, metric, claim_scope,
                placement='Main text', data=None):
    stem = f'{figure_id}_{re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_").lower()}'
    output_dir = APPENDIX_DIR if figure_id.startswith('Appendix_') else FIGURE_DIR
    paths = {}
    for ext in ['png', 'pdf', 'svg']:
        path = output_dir / f'{stem}.{ext}'
        fig.savefig(path, dpi=400 if ext == 'png' else None, bbox_inches='tight', facecolor='white')
        paths[ext] = str(path)
    data_path = ''
    if data is not None:
        data_path = TABLE_DIR / f'{stem}_data.csv'
        data.to_csv(data_path, index=False)
    figure_records.append({
        'figure_id': figure_id, 'title': title, 'chapter_or_rq': chapter,
        'source_files': '; '.join(map(str, source_files)), 'metric': metric,
        'claim_scope': claim_scope, 'recommended_placement': placement,
        'png': paths['png'], 'pdf': paths['pdf'], 'svg': paths['svg'],
        'plotted_data_csv': str(data_path) if data_path else '',
    })
    plt.show(); plt.close(fig)

def panel_label(ax, label):
    ax.text(-.045, 1.025, label, transform=ax.transAxes, fontsize=11,
            fontweight='bold', va='bottom', ha='right', color=COLORS['ink'])

def numeric(df, columns):
    out = df.copy()
    for c in columns:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')
    return out

def require_files(paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise FileNotFoundError('Missing frozen source files:\n' + '\n'.join(missing))

def add_map_context(ax, add_basemap=True, attribution=False, fill=False):
    if boroughs is not None:
        xmin, ymin, xmax, ymax = boroughs.total_bounds
        padx, pady = (xmax-xmin)*.025, (ymax-ymin)*.025
        ax.set_xlim(xmin-padx, xmax+padx); ax.set_ylim(ymin-pady, ymax+pady)
        if add_basemap and USE_BASEMAP:
            try:
                ctx.add_basemap(ax, crs='EPSG:27700', source=ctx.providers.CartoDB.PositronNoLabels,
                                alpha=.66, attribution=False, zorder=0)
            except Exception as exc:
                warnings.warn(f'Basemap unavailable; continuing with boundary-only map: {exc}')
        if fill:
            boroughs.plot(ax=ax, facecolor='#E9E8E3', edgecolor='white', linewidth=.45, alpha=.76, zorder=1)
        boroughs.boundary.plot(ax=ax, color='#6F767A', linewidth=.34, alpha=.82, zorder=4)
        boroughs.dissolve().boundary.plot(ax=ax, color='#252A2D', linewidth=.85, zorder=5)
    ax.set_axis_off(); ax.set_aspect('equal')
    if attribution:
        ax.text(.995,.005,'Basemap: CARTO / OpenStreetMap contributors',transform=ax.transAxes,
                ha='right',va='bottom',fontsize=5.7,color='#73787C')

def add_north_scale(ax, length_km=10):
    xmin,xmax=ax.get_xlim(); ymin,ymax=ax.get_ylim()
    x0=xmin+.055*(xmax-xmin); y0=ymin+.055*(ymax-ymin); length=length_km*1000
    ax.plot([x0,x0+length],[y0,y0],color=COLORS['ink'],lw=1.6,zorder=10)
    ax.plot([x0,x0],[y0-320,y0+320],color=COLORS['ink'],lw=.9,zorder=10)
    ax.plot([x0+length,x0+length],[y0-320,y0+320],color=COLORS['ink'],lw=.9,zorder=10)
    ax.text(x0+length/2,y0+600,f'{length_km} km',ha='center',va='bottom',fontsize=6.3,zorder=10)
    ax.annotate('N',xy=(.94,.14),xytext=(.94,.06),xycoords='axes fraction',ha='center',va='center',
                fontsize=7.3,fontweight='bold',arrowprops=dict(arrowstyle='-|>',color=COLORS['ink'],lw=.9))

def colour_for_delta(value):
    return COLORS['positive'] if value > .002 else (COLORS['negative'] if value < -.002 else COLORS['mid'])

def dataframe_to_markdown(frame):
    clean=frame.fillna('').astype(str).applymap(lambda v:v.replace('|','\\|').replace('\n',' '))
    header='| '+' | '.join(clean.columns)+' |'
    rule='| '+' | '.join(['---']*len(clean.columns))+' |'
    rows=['| '+' | '.join(row)+' |' for row in clean.to_numpy().tolist()]
    return '\n'.join([header,rule]+rows)

print('Structural reporting directory:', REPORT_DIR)


## Frozen inputs and readiness gate

The readiness gate checks the canonical sample table, fixed validation assignments, OOF predictions and frozen summaries. London-wide clean PTAL/EPC pools are loaded only to describe sampling representativeness. Image archives are used only for reader-facing input examples.


In [ ]:
LEGACY_TABLE_DIR = cfg.TABLE_DIR
RAW_STREET_ZIP = cfg.RAW_DIR / 'Street_view' / 'Streetviews.zip'
AERIAL_SOURCE_INDEX = cfg.AERIAL_CROP_SOURCE_INDEX_PATH
STREET_IMAGE_INVENTORY = LEGACY_TABLE_DIR / 'streetview_image_inventory.csv'
DINO_SATCLIP_COMPARISON = cfg.IMAGE_LOCATION_COMPARISON_SUMMARY_PATH
EPC_FULL_CORRECTED = cfg.AUDIT_DIR / 'epc_postcode_corrected_candidate_v2.parquet'

SOURCE_PATHS = {
    'model_table': cfg.FINAL_MODEL_TABLE_PATH,
    'ptal_clean_pool': cfg.LEGACY_PTAL_CLEAN,
    'epc_corrected_candidate_pool': EPC_FULL_CORRECTED,
    'borough_folds': cfg.RIDGE_OUTER_FOLDS_PATH,
    'random_folds': cfg.RANDOM_OUTER_FOLDS_PATH,
    'geometric_folds': cfg.GEOMETRIC_OUTER_FOLDS_PATH,
    '06_core_folds': cfg.RIDGE_CORE_RESULTS_PATH,
    '07_incremental_folds': cfg.INCREMENTAL_RESULTS_PATH,
    '07_paired_deltas': cfg.INCREMENTAL_DELTAS_PATH,
    '07_control_ablation': cfg.INCREMENTAL_ABLATION_SUMMARY_PATH,
    '07_predictions': cfg.INCREMENTAL_PREDICTIONS_PATH,
    '09_dino_satclip': DINO_SATCLIP_COMPARISON,
    '10_pca_comparison': cfg.PCA64_COMPARISON_SUMMARY_PATH,
    '11_protocol_summary': cfg.GEOMETRIC_PROTOCOL_SUMMARY_PATH,
    '11_geometric_predictions': cfg.GEOMETRIC_CV_PREDICTIONS_PATH,
    '11_residual_moran': cfg.GEOMETRIC_RESIDUAL_MORAN_PATH,
    '12_xgb_vs_ridge': cfg.XGBOOST_VS_RIDGE_SUMMARY_PATH,
    '13_gat_vs_mlp': cfg.GATV2_VS_MLP_SUMMARY_PATH,
    '14_epc_protocols': cfg.EPC_ROBUST_INCREMENTAL_SUMMARY_PATH,
    '14_epc_year_bins': cfg.EPC_TEMPORAL_BIN_PATH,
    'provenance': cfg.PROVENANCE_TABLE_PATH,
    'aerial_source_index': AERIAL_SOURCE_INDEX,
    'street_image_inventory': STREET_IMAGE_INVENTORY,
    'street_image_archive': RAW_STREET_ZIP,
}
readiness=pd.DataFrame([{'source':k,'path':str(v),'available':Path(v).exists(),
                         'size_mb':round(Path(v).stat().st_size/1e6,3) if Path(v).exists() else np.nan}
                        for k,v in SOURCE_PATHS.items()])
display(readiness); require_files(SOURCE_PATHS.values())
readiness.to_csv(MANIFEST_DIR/'source_readiness.csv',index=False)

core_fold=numeric(pd.read_csv(cfg.RIDGE_CORE_RESULTS_PATH),['outer_fold','r2','rmse','mae'])
inc_fold=numeric(pd.read_csv(cfg.INCREMENTAL_RESULTS_PATH),['outer_fold','r2','rmse','mae'])
paired_delta=numeric(pd.read_csv(cfg.INCREMENTAL_DELTAS_PATH),['outer_fold','delta_r2','delta_rmse','delta_mae'])
control_ablation=numeric(pd.read_csv(cfg.INCREMENTAL_ABLATION_SUMMARY_PATH),['mean_delta_r2','sd_delta_r2'])
protocol_summary=numeric(pd.read_csv(cfg.GEOMETRIC_PROTOCOL_SUMMARY_PATH),['mean_r2','sd_r2','pooled_r2','pooled_rmse','pooled_mae'])
moran_summary=numeric(pd.read_csv(cfg.GEOMETRIC_RESIDUAL_MORAN_PATH),['moran_i','permutation_p_two_sided'])
xgb_vs_ridge=numeric(pd.read_csv(cfg.XGBOOST_VS_RIDGE_SUMMARY_PATH),['xgboost_pooled_r2','ridge_pooled_r2','pooled_delta_r2','mean_delta_r2','sd_delta_r2'])
pca_comparison=numeric(pd.read_csv(cfg.PCA64_COMPARISON_SUMMARY_PATH),['mean_delta_r2','sd_delta_r2'])
gat_vs_mlp=numeric(pd.read_csv(cfg.GATV2_VS_MLP_SUMMARY_PATH),['mean_delta_r2_gatv2_minus_mlp','sd_delta_r2'])
epc_protocols=numeric(pd.read_csv(cfg.EPC_ROBUST_INCREMENTAL_SUMMARY_PATH),['mean_delta_r2','sd_delta_r2'])
epc_year_bins=numeric(pd.read_csv(cfg.EPC_TEMPORAL_BIN_PATH),['n','mean_record_year','mean_signed_error','mean_absolute_error','rmse'])
dino_satclip=numeric(pd.read_csv(DINO_SATCLIP_COMPARISON),['mean_delta_r2','sd_delta_r2'])

print('Frozen reporting sources: READY')


## Spatial tables, full comparison pools and map context

Only identifiers, target values, coordinates and a few descriptive fields are read from the wide feature table. The original 26,597 rows and all fixed fold assignments remain unchanged.


In [ ]:
import pyarrow.parquet as pq

schema_cols=pq.ParquetFile(cfg.FINAL_MODEL_TABLE_PATH).schema.names
wanted=['sample_id','task','target','borough_code','x','y','lon','lat','n_properties',
        'median_record_year','sv_has_streetview','sv_n_images']
map_cols=[c for c in wanted if c in schema_cols]
map_df=pd.read_parquet(cfg.FINAL_MODEL_TABLE_PATH,columns=map_cols)
map_df['sample_id']=map_df['sample_id'].astype(str)
for c in ['target','x','y','lon','lat','n_properties','median_record_year','sv_has_streetview','sv_n_images']:
    if c in map_df: map_df[c]=pd.to_numeric(map_df[c],errors='coerce')

if {'x','y'}.issubset(map_df.columns):
    points=gpd.GeoDataFrame(map_df.copy(),geometry=gpd.points_from_xy(map_df.x,map_df.y),crs='EPSG:27700')
elif {'lon','lat'}.issubset(map_df.columns):
    points=gpd.GeoDataFrame(map_df.copy(),geometry=gpd.points_from_xy(map_df.lon,map_df.lat),crs='EPSG:4326').to_crs('EPSG:27700')
else:
    raise KeyError('No recognised coordinate pair in final model table.')

def find_london_boundaries():
    candidates=[]
    for suffix in ['*.gpkg','*.geojson','*.shp']:
        candidates.extend(cfg.BOUNDARY_DIR.rglob(suffix))
    candidates=sorted(candidates,key=lambda p:(not any(k in p.name.lower() for k in ['borough','lad','london']),len(str(p))))
    for path in candidates:
        try:
            g=gpd.read_file(path)
            if g.empty: continue
            code_col=next((c for c in ['borough_code','GSS_CODE','LAD23CD','LAD22CD','LAD21CD','LAD19CD'] if c in g),None)
            if code_col:
                subset=g[g[code_col].astype(str).isin(cfg.LONDON_BOROUGH_CODE_TO_NAME)].copy()
                if len(subset)>=30: return subset.to_crs('EPSG:27700'),code_col,path
            if 20<=len(g)<=40: return g.to_crs('EPSG:27700'),code_col,path
        except Exception: continue
    return None,None,None

boroughs,borough_code_col,boundary_source=find_london_boundaries()
if boroughs is None: raise FileNotFoundError('London borough boundaries are required for the redesigned maps.')

borough_folds=pd.read_csv(cfg.RIDGE_OUTER_FOLDS_PATH)
random_folds=pd.read_csv(cfg.RANDOM_OUTER_FOLDS_PATH)
geometric_folds=pd.read_csv(cfg.GEOMETRIC_OUTER_FOLDS_PATH)
for frame in [borough_folds,random_folds,geometric_folds]: frame['sample_id']=frame['sample_id'].astype(str)

def read_target_pool(path, task):
    df=pd.read_parquet(path) if Path(path).suffix.lower() in ['.parquet','.pq'] else pd.read_csv(path,low_memory=False)
    candidates=['target','label_regression','ptal','ptal_value','mean_ptal','current_energy_efficiency']
    col=next((c for c in candidates if c in df.columns),None)
    if col is None:
        likely=[c for c in df.columns if any(k in c.lower() for k in ['target','ptal','energy_efficiency'])]
        col=next((c for c in likely if pd.api.types.is_numeric_dtype(df[c])),None)
    if col is None: raise KeyError(f'No target column found in {path.name}')
    return pd.DataFrame({'task':task,'target':pd.to_numeric(df[col],errors='coerce')}).dropna()

full_pool=pd.concat([read_target_pool(cfg.LEGACY_PTAL_CLEAN,'PTAL'),read_target_pool(EPC_FULL_CORRECTED,'EPC')],ignore_index=True)
print('Analytical sample:',points.task.value_counts().to_dict())
print('London-wide comparison pools:',full_pool.task.value_counts().to_dict())


## Reusable real-image assets

One fixed central-London anchor is chosen by a deterministic coverage rule. The same centre is used for the 150 m and 300 m aerial crops. Street-view thumbnails are the nearest readable archived images within the task-specific radii; outcomes and model performance are never consulted.


In [ ]:
def parse_tile_paths(row):
    candidates=[]
    for col in ['tile_paths','tile_path']:
        value=row.get(col,'')
        if pd.isna(value) or not str(value).strip(): continue
        value=str(value).strip()
        try:
            parsed=ast.literal_eval(value); candidates.extend(parsed if isinstance(parsed,(list,tuple)) else [parsed])
        except Exception:
            candidates.extend([v.strip() for v in re.split(r'[|;]',value) if v.strip()])
    return list(dict.fromkeys(Path(p) for p in candidates if Path(p).exists()))

def aerial_crop_from_row(row,bounds,output_px=760):
    import rasterio
    from rasterio.enums import Resampling
    from rasterio.merge import merge
    from rasterio.windows import from_bounds
    paths=parse_tile_paths(row)
    if not paths: raise FileNotFoundError('No indexed Digimap tile for selected anchor.')
    datasets=[rasterio.open(p) for p in paths]
    try:
        if len(datasets)==1:
            src=datasets[0]; window=from_bounds(*bounds,transform=src.transform).round_offsets().round_lengths()
            bands=list(range(1,min(src.count,3)+1))
            arr=src.read(bands,window=window,out_shape=(len(bands),output_px,output_px),
                         resampling=Resampling.bilinear,boundless=True,fill_value=255)
        else:
            arr,_=merge(datasets,bounds=bounds,indexes=list(range(1,min(datasets[0].count,3)+1)))
        arr=np.moveaxis(arr[:3],0,-1)
        if arr.shape[-1]==1: arr=np.repeat(arr,3,axis=-1)
        if arr.dtype!=np.uint8:
            lo,hi=np.nanpercentile(arr,[1,99]); arr=np.clip((arr-lo)/(hi-lo+1e-9)*255,0,255).astype(np.uint8)
        return np.asarray(ImageOps.fit(Image.fromarray(arr).convert('RGB'),(output_px,output_px),method=Image.Resampling.LANCZOS))
    finally:
        for ds in datasets: ds.close()

def panel_square(image,output_px=500):
    image=image if isinstance(image,Image.Image) else Image.fromarray(image)
    return np.asarray(ImageOps.fit(ImageOps.exif_transpose(image).convert('RGB'),(output_px,output_px),method=Image.Resampling.LANCZOS))

aerial_index=pd.read_csv(AERIAL_SOURCE_INDEX,low_memory=False)
for c in ['x','y']:
    aerial_index[c]=pd.to_numeric(aerial_index[c],errors='coerce')
aerial_index['_central_distance']=np.hypot(aerial_index.x-cfg.LONDON_CENTRE_EASTING,
                                            aerial_index.y-cfg.LONDON_CENTRE_NORTHING)

street_inventory=pd.read_csv(STREET_IMAGE_INVENTORY,low_memory=False)
street_inventory['_x']=pd.to_numeric(street_inventory['sv_x'],errors='coerce')
street_inventory['_y']=pd.to_numeric(street_inventory['sv_y'],errors='coerce')
if street_inventory['_x'].abs().median()<10000:
    sg=gpd.GeoSeries(gpd.points_from_xy(street_inventory._x,street_inventory._y),crs='EPSG:4326').to_crs('EPSG:27700')
    street_inventory['_x']=sg.x; street_inventory['_y']=sg.y
street_inventory=street_inventory.dropna(subset=['_x','_y']).copy()
if 'has_image' in street_inventory:
    street_inventory=street_inventory[pd.to_numeric(street_inventory.has_image,errors='coerce').eq(1)].copy()

with zipfile.ZipFile(RAW_STREET_ZIP) as archive:
    members=set(archive.namelist())
street_inventory=street_inventory[street_inventory.image_in_zip.astype(str).isin(members)].copy()
coords=street_inventory[['_x','_y']].to_numpy()
tree=BallTree(coords,leaf_size=40)
central_order=np.argsort(np.hypot(coords[:,0]-cfg.LONDON_CENTRE_EASTING,coords[:,1]-cfg.LONDON_CENTRE_NORTHING))[:4000]
anchor_idx=None
for idx in central_order:
    n150=tree.query_radius(coords[idx:idx+1],r=150,count_only=True)[0]
    n300=tree.query_radius(coords[idx:idx+1],r=300,count_only=True)[0]
    if n150>=4 and n300>=8:
        anchor_idx=idx; break
if anchor_idx is None: raise RuntimeError('No street-image anchor met the frozen 4/150 m and 8/300 m support rules.')
anchor_x,anchor_y=coords[anchor_idx]

def nearest_street_images(radius,k):
    ind,dist=tree.query_radius([[anchor_x,anchor_y]],r=radius,return_distance=True,sort_results=True)
    chosen=[]
    with zipfile.ZipFile(RAW_STREET_ZIP) as archive:
        for idx,d in zip(ind[0],dist[0]):
            row=street_inventory.iloc[int(idx)]
            try:
                im=Image.open(BytesIO(archive.read(str(row.image_in_zip)))).convert('RGB')
                chosen.append((panel_square(im,360),float(d),str(row.image_in_zip)))
            except Exception: continue
            if len(chosen)==k: break
    if len(chosen)<k: raise RuntimeError(f'Only {len(chosen)} readable street images within {radius} m; expected {k}.')
    return chosen

ptal_street=nearest_street_images(300,8)
epc_street=nearest_street_images(150,4)

# Use an indexed aerial row nearest the same Street View anchor; try nearby rows until both crops read.
aerial_index['_anchor_distance']=np.hypot(aerial_index.x-anchor_x,aerial_index.y-anchor_y)
for _,candidate in aerial_index.sort_values('_anchor_distance').head(200).iterrows():
    try:
        centre_x=float(candidate.x); centre_y=float(candidate.y)
        aerial_300=aerial_crop_from_row(candidate,(centre_x-150,centre_y-150,centre_x+150,centre_y+150))
        aerial_150=aerial_crop_from_row(candidate,(centre_x-75,centre_y-75,centre_x+75,centre_y+75))
        aerial_example_row=candidate; break
    except Exception: continue
else: raise RuntimeError('Could not read comparable 150 m and 300 m aerial crops.')

image_asset_manifest=pd.DataFrame([
    {'asset':'aerial_300','centre_x':centre_x,'centre_y':centre_y,'support_m':300,'n_images':1},
    {'asset':'aerial_150','centre_x':centre_x,'centre_y':centre_y,'support_m':150,'n_images':1},
    {'asset':'street_ptal','centre_x':anchor_x,'centre_y':anchor_y,'support_m':300,'n_images':8},
    {'asset':'street_epc','centre_x':anchor_x,'centre_y':anchor_y,'support_m':150,'n_images':4},
])
image_asset_manifest.to_csv(MANIFEST_DIR/'image_asset_manifest.csv',index=False)
display(image_asset_manifest)


## Figure 1. Detailed end-to-end workflow

The workflow is intentionally information-dense: it links sample construction, five frozen encoders, task-specific image support, matched controls, nested spatial validation, robustness branches and the evidence used in the dissertation.


In [ ]:
def draw_database(ax,x,y,w,h,colour):
    ax.add_patch(patches.Rectangle((x,y),w,h,facecolor=colour,edgecolor='white',lw=1))
    ax.add_patch(patches.Ellipse((x+w/2,y+h),w,h*.22,facecolor=colour,edgecolor='white',lw=1))
    ax.add_patch(patches.Ellipse((x+w/2,y),w,h*.22,facecolor=colour,edgecolor='white',lw=1))
    for f in [.35,.68]: ax.plot([x,x+w],[y+h*f,y+h*f],color='white',lw=.7,alpha=.8)

def arrow(ax,x1,y1,x2,y2):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='-|>',lw=1.15,color='#747B80'))

if BUILD_MAIN_FIGURES:
    fig,ax=plt.subplots(figsize=(16.2,9.2)); ax.set_xlim(0,16.2); ax.set_ylim(0,9.2); ax.axis('off')
    # Column guides and stage headings.
    for x in [3.45,8.25,12.25]: ax.plot([x,x],[.35,8.55],color='#E0E2E2',lw=.8)
    for x,t in [(1.65,'DATA & SAMPLES'),(5.8,'FROZEN REPRESENTATIONS'),(10.2,'MODELLING & VALIDATION'),(14.2,'EVIDENCE')]:
        ax.text(x,8.85,t,ha='center',va='center',fontsize=10.5,fontweight='bold',color='#3D4448')

    # PTAL and EPC databases plus sample-construction steps.
    draw_database(ax,.35,6.65,.62,.75,COLORS['individual']); ax.text(.66,6.25,'TfL PTAL 2023\n157,322 clean points',ha='center',fontsize=7.7)
    ax.text(2.05,7.0,'500 m modelling grid',ha='center',va='center',fontsize=7.8,fontweight='semibold')
    ax.add_patch(patches.Circle((3.0,7.0),.34,facecolor='white',edgecolor=COLORS['individual'],lw=1.5)); ax.text(3.0,7.0,'6,597',ha='center',va='center',fontsize=8,fontweight='bold')
    arrow(ax,.98,7.0,1.55,7.0); arrow(ax,2.53,7.0,2.64,7.0)

    draw_database(ax,.35,3.75,.62,.75,COLORS['fusion']); ax.text(.66,3.2,'EPC certificates\n3,495,691 records',ha='center',fontsize=7.7)
    ax.add_patch(patches.FancyBboxPatch((1.20,3.63),1.78,1.0,boxstyle='round,pad=.03',facecolor='#F3F1EC',edgecolor='#A8ADB0',lw=.75))
    ax.text(2.09,4.13,'UPRN/address resolution\n→ latest certificate\n→ postcode mean',ha='center',va='center',fontsize=7.05,linespacing=1.25)
    arrow(ax,.98,4.13,1.15,4.13)
    ax.text(2.15,3.18,'149,471 candidate postcodes\nlocal-authority area × EPC stratification',ha='center',fontsize=7.15)
    ax.add_patch(patches.Circle((3.0,2.55),.34,facecolor='white',edgecolor=COLORS['fusion'],lw=1.5)); ax.text(3.0,2.55,'20,000',ha='center',va='center',fontsize=7.7,fontweight='bold')
    arrow(ax,2.8,3.1,2.98,2.91)
    ax.add_patch(patches.FancyBboxPatch((.52,.72),2.45,.72,boxstyle='round,pad=.04',facecolor='#ECEDEA',edgecolor='#7A8084',lw=1))
    ax.text(1.745,1.08,'Representation-complete sample\nn = 26,597',ha='center',va='center',fontsize=8.2,fontweight='bold')
    ax.plot([3.0,3.28,3.28],[6.63,6.25,1.62],color='#747B80',lw=1.15);arrow(ax,3.28,1.62,2.62,1.45);arrow(ax,3.0,2.18,2.55,1.45)

    # Five encoders: actual image tiles where available plus compact vector blocks.
    rep_x=[3.78,4.67,5.56,6.45,7.34]
    rep=[('Coordinates','SatCLIP','256-D'),('EO time series','TESSERA','128-D'),('Annual EO','AlphaEarth','64-D'),('Aerial 25 cm','DINOv2','768-D'),('Street View','CLIP','512-D')]
    for j,(x,(source,encoder,dim)) in enumerate(zip(rep_x,rep)):
        if j==3: ax.imshow(aerial_300,extent=(x,x+.72,6.55,7.27),zorder=2)
        elif j==4: ax.imshow(ptal_street[0][0],extent=(x,x+.72,6.55,7.27),zorder=2)
        else:
            ax.add_patch(patches.Rectangle((x,6.55),.72,.72,facecolor=['#DCE7EA','#CADDE3','#D9E1D1'][j],edgecolor='white'))
            for k in range(4): ax.add_patch(patches.Rectangle((x+.08+k*.15,6.66),.085,.42,facecolor=COLORS['individual'],alpha=.25+.16*k,edgecolor='none'))
        ax.text(x+.36,6.33,source,ha='center',fontsize=6.8); ax.text(x+.36,6.05,encoder,ha='center',fontsize=7.4,fontweight='bold'); ax.text(x+.36,5.78,dim,ha='center',fontsize=7.1,color=COLORS['mid'])
    ax.text(5.8,7.62,'Five pretrained encoders',ha='center',fontsize=9.3,fontweight='bold')

    # Task support and embedding fusion.
    ax.add_patch(patches.Rectangle((3.95,3.58),1.25,1.25,fill=False,edgecolor=COLORS['individual'],lw=1.8)); ax.text(4.575,4.2,'PTAL\n300 m aerial\n≤8 street images / 300 m',ha='center',va='center',fontsize=7.5,color=COLORS['individual'],fontweight='semibold')
    ax.add_patch(patches.Rectangle((5.45,3.89),.63,.63,fill=False,edgecolor=COLORS['fusion'],lw=1.8)); ax.text(5.765,3.48,'EPC\n150 m aerial\n≤4 street images / 150 m',ha='center',va='top',fontsize=7.5,color=COLORS['fusion'],fontweight='semibold')
    ax.text(6.85,4.2,'[ 256 | 128 | 64 | 768 | 512 ]',ha='center',fontsize=8.2,family='monospace')
    ax.text(6.85,3.72,'single representations and\npre-specified combinations',ha='center',fontsize=7.4,color=COLORS['mid'])
    for x in rep_x: arrow(ax,x+.36,5.65,6.75,4.48)
    arrow(ax,2.98,1.08,4.4,3.5)

    # Modelling and nested CV mini-schematics.
    ax.text(9.15,7.55,'Matched controls',ha='center',fontsize=9,fontweight='bold')
    ax.add_patch(patches.FancyBboxPatch((8.7,6.5),1.0,.72,boxstyle='round,pad=.03',facecolor='#ECEDEA',edgecolor='#7B8286')); ax.text(9.2,6.86,'PTAL spatial\ncontrols',ha='center',va='center',fontsize=7.3)
    ax.add_patch(patches.FancyBboxPatch((9.82,6.5),1.0,.72,boxstyle='round,pad=.03',facecolor='#ECEDEA',edgecolor='#7B8286')); ax.text(10.32,6.86,'EPC compact /\nprivileged extended',ha='center',va='center',fontsize=6.8)
    ax.add_patch(patches.FancyBboxPatch((9.1,4.75),1.55,.85,boxstyle='round,pad=.04',facecolor='#F2EFE9',edgecolor=COLORS['fusion'],lw=1.2)); ax.text(9.875,5.18,'Primary linear probe\nRidge regression',ha='center',va='center',fontsize=8.3,fontweight='bold')
    arrow(ax,7.75,4.2,9.0,5.0); arrow(ax,9.2,6.45,9.55,5.62); arrow(ax,10.3,6.45,10.1,5.62)
    ax.text(10.15,3.95,'Nested spatial validation',ha='center',fontsize=9,fontweight='bold')
    for i,c in enumerate(FOLD_COLORS):
        ax.add_patch(patches.Rectangle((8.62+i*.36,3.28),.29,.42,facecolor=c,edgecolor='white',lw=.5))
    ax.text(9.52,2.92,'outer 5-fold\nborough-level hold-out',ha='center',va='top',fontsize=6.85)
    for i in range(3): ax.add_patch(patches.Rectangle((10.58+i*.35,3.28),.28,.42,facecolor='#B7BEC2',edgecolor='white'))
    ax.text(10.93,2.92,'inner 3-fold\ngroup-aware tuning',ha='center',va='top',fontsize=6.85)
    ax.text(10.15,2.38,'fold-local imputation and scaling',ha='center',fontsize=7.5,color=COLORS['mid'])

    # Robustness branch uses compact icons rather than a text-only panel.
    ax.text(10.15,1.62,'Robustness branches',ha='center',fontsize=8.7,fontweight='bold')
    robustness=['Random CV','Continuous CV','XGBoost','PCA64','MLP / GATv2','EPC reliability / timing']
    for i,label in enumerate(robustness):
        x=8.45+(i%3)*1.15; y=.72+(i//3)*.48
        ax.add_patch(patches.FancyBboxPatch((x,y),1.03,.32,boxstyle='round,pad=.02',facecolor='#F4F3EF',edgecolor='#A2A7AA',lw=.65)); ax.text(x+.515,y+.16,label,ha='center',va='center',fontsize=6.7)

    # Evidence column. OOF predictions are the common source for performance
    # summaries and residual diagnostics; ΔR² compares matched model metrics.
    evidence_boxes=[
        (13.05,6.75,2.35,.72,'OOF predictions'),
        (12.58,5.18,1.48,.72,'R²  RMSE  MAE'),
        (14.22,5.18,1.48,.72,'incremental ΔR²'),
        (12.58,3.55,1.48,.72,'residual\ngeography'),
        (12.58,1.95,1.48,.72,"Moran's I"),
    ]
    for x,y,w,h,label in evidence_boxes:
        ax.add_patch(patches.FancyBboxPatch((x,y),w,h,boxstyle='round,pad=.03',facecolor='#F4F3EF',edgecolor='#858C90',lw=.8))
        ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=7.8,fontweight='semibold')
    arrow(ax,11.2,5.15,13.0,7.02)
    arrow(ax,13.78,6.73,13.32,5.93); arrow(ax,14.68,6.73,14.96,5.93)
    arrow(ax,13.34,6.73,13.32,4.31); arrow(ax,13.32,3.52,13.32,2.70)
    fig.tight_layout()
    workflow_data=pd.DataFrame([
        ['PTAL_clean',157322],['PTAL_final',6597],['EPC_raw_certificates',3495691],
        ['EPC_candidate_postcodes',149471],['EPC_final',20000],['representation_complete_total',26597]
    ],columns=['stage','n'])
    save_figure(fig,'Figure_1','detailed end to end workflow','Methods',
                ['notebooks 01–14',cfg.FEATURE_MANIFEST_JSON_PATH,AERIAL_SOURCE_INDEX,STREET_IMAGE_INVENTORY],
                'Data flow, representation dimensions, validation design and evidence',
                'Schematic of the frozen design; icons and thumbnails do not encode model performance.',data=workflow_data)


## Figure 2. Study area and actual analytical samples

The first map locates Greater London and the River Thames. The second and third maps show the original PTAL and EPC analytical rows directly—no hexagonal aggregation is used.


In [ ]:
if BUILD_MAIN_FIGURES:
    fig,axes=plt.subplots(1,3,figsize=(15.7,5.7))
    add_map_context(axes[0],add_basemap=True,attribution=True,fill=True)
    axes[0].set_title('Greater London',loc='left')
    axes[0].text(532000,177300,'River Thames',fontsize=7,color='#567D91',rotation=-8,
                 path_effects=[pe.withStroke(linewidth=2.2,foreground='white')],zorder=8)
    add_north_scale(axes[0],10)
    try:
        ins=inset_axes(axes[0],width='29%',height='29%',loc='upper left',borderpad=.8)
        uk_box=gpd.GeoSeries([gpd.points_from_xy([-8.8],[49.6])[0],gpd.points_from_xy([2.0],[59.2])[0]],crs='EPSG:4326').to_crs('EPSG:3857')
        ins.set_xlim(uk_box.x.iloc[0],uk_box.x.iloc[1]);ins.set_ylim(uk_box.y.iloc[0],uk_box.y.iloc[1])
        ctx.add_basemap(ins,source=ctx.providers.CartoDB.PositronNoLabels,zoom=5,attribution=False)
        london=gpd.GeoSeries(gpd.points_from_xy([-.1276],[51.5072]),crs='EPSG:4326').to_crs('EPSG:3857')
        ins.scatter(london.x,london.y,s=18,color=COLORS['negative'],edgecolor='white',lw=.5,zorder=5)
        ins.set_xticks([]);ins.set_yticks([]);ins.set_title('London in the UK',fontsize=6.2,pad=2)
    except Exception as exc: warnings.warn(f'Locator inset unavailable: {exc}')

    plotted=[]
    for ax,task,cmap,title,size,alpha in [
        (axes[1],'PTAL','viridis','PTAL analytical points',5.8,.86),
        (axes[2],'EPC','cividis','EPC postcode sample',2.0,.55),
    ]:
        add_map_context(ax,add_basemap=True,attribution=False)
        g=points[points.task.eq(task)].copy()
        sc=ax.scatter(g.geometry.x,g.geometry.y,c=g.target,s=size,alpha=alpha,cmap=cmap,
                      linewidths=0,rasterized=True,zorder=2)
        if boroughs is not None: boroughs.boundary.plot(ax=ax,color='#4F565A',lw=.28,alpha=.65,zorder=4)
        ax.set_title(f'{title}  (n = {len(g):,})',loc='left')
        target_label='Access Index' if task=='PTAL' else 'Mean current-efficiency score'
        cbar=fig.colorbar(sc,ax=ax,fraction=.037,pad=.012);cbar.set_label(target_label,fontsize=7.5);cbar.ax.tick_params(labelsize=7)
        plotted.extend(g[['sample_id','task','target','borough_code']].to_dict('records'))
    for i,ax in enumerate(axes): panel_label(ax,chr(97+i))
    fig.tight_layout(w_pad=.6)
    save_figure(fig,'Figure_2','study area and actual analytical samples','Methods',
                [cfg.FINAL_MODEL_TABLE_PATH,boundary_source],
                'Observed target at original analytical locations',
                'Point opacity is cartographic only; no aggregation or resampling is applied.',data=pd.DataFrame(plotted))


## Figure 3. Outcome distributions and sampling representativeness

Normalised distributions compare each London-wide clean pool with the final representation-complete analytical sample. Common bin edges and descriptive mean/median markers make any target-distribution shift visible without a significance claim.


In [ ]:
if BUILD_MAIN_FIGURES:
    fig,axes=plt.subplots(1,2,figsize=(12.8,4.8)); plotted=[]
    for ax,task in zip(axes,['PTAL','EPC']):
        full=full_pool.loc[full_pool.task.eq(task),'target'].dropna().to_numpy()
        sample=map_df.loc[map_df.task.eq(task),'target'].dropna().to_numpy()
        lo=min(np.nanpercentile(full,.25),np.nanpercentile(sample,.25));hi=max(np.nanpercentile(full,99.75),np.nanpercentile(sample,99.75))
        bins=np.linspace(lo,hi,36)
        ax.hist(full,bins=bins,density=True,histtype='stepfilled',alpha=.32,color='#AEB5B9',label=f'London-wide clean pool (n={len(full):,})')
        ax.hist(sample,bins=bins,density=True,histtype='step',lw=1.8,color=COLORS['individual'],label=f'Analytical sample (n={len(sample):,})')
        ax.axvline(np.nanmedian(full),color='#7A8185',ls=':',lw=1)
        ax.axvline(np.nanmedian(sample),color=COLORS['individual'],ls='--',lw=1)
        target_label='Access Index' if task=='PTAL' else 'Mean current-efficiency score'
        ax.set(xlabel=target_label,ylabel='Density',title=task)
        ax.legend(frameon=False,loc='upper right')
        ax.text(.02,.95,f'Pool mean / median: {np.mean(full):.2f} / {np.median(full):.2f}\nSample mean / median: {np.mean(sample):.2f} / {np.median(sample):.2f}',transform=ax.transAxes,va='top',fontsize=7.5)
        plotted.extend(pd.DataFrame({'task':task,'source':'London-wide clean pool','target':full}).to_dict('records'))
        plotted.extend(pd.DataFrame({'task':task,'source':'Analytical sample','target':sample}).to_dict('records'))
    panel_label(axes[0],'a');panel_label(axes[1],'b');fig.tight_layout(w_pad=2)
    save_figure(fig,'Figure_3','outcome distributions and sampling representativeness','Methods',
                [cfg.LEGACY_PTAL_CLEAN,EPC_FULL_CORRECTED,cfg.FINAL_MODEL_TABLE_PATH],
                'Normalised target density; descriptive mean and median',
                'Distribution comparison is descriptive and does not establish probabilistic representativeness.',data=pd.DataFrame(plotted))


## Figure 4. Spatial validation design

Rows distinguish PTAL and EPC; columns distinguish random folds, held-out borough groups and compact continuous regions. This removes validation geography from the study-area figure and makes the deployment assumptions explicit.


In [ ]:
if BUILD_MAIN_FIGURES:
    fig,axes=plt.subplots(2,3,figsize=(14.3,9.0)); plotted=[]
    schemes=[('Random folds',random_folds,'random'),('Borough-level held-out groups',borough_folds,'borough'),('Continuous-region folds',geometric_folds,'geometric')]

    # The three frozen protocols were created by different notebooks and use
    # protocol-specific names for the same concept.  Resolve the source column
    # explicitly, then map its five labels to 0--4 only for plotting colours.
    fold_candidates={
        'random':['random_outer_fold','random_fold','outer_fold','test_fold','fold','split_id'],
        'borough':['outer_fold','borough_outer_fold','borough_fold','test_fold','fold'],
        'geometric':['geometric_outer_fold','geometric_fold','region_fold','outer_fold','test_fold','fold'],
    }

    def normalise_fold_assignment(frame,kind):
        candidates=fold_candidates[kind]
        source_col=next((c for c in candidates if c in frame.columns),None)
        if source_col is None:
            fallback=[c for c in frame.columns if 'fold' in c.lower() and 'inner' not in c.lower()]
            if len(fallback)==1: source_col=fallback[0]
        if source_col is None:
            raise KeyError(f'No outer-fold column recognised for {kind}. Available columns: {list(frame.columns)}')
        if frame[source_col].isna().any():
            raise ValueError(f'{kind} fold assignment contains missing values in {source_col}.')
        numeric_values=pd.to_numeric(frame[source_col],errors='coerce')
        values=numeric_values if numeric_values.notna().all() else frame[source_col].astype(str)
        labels=sorted(pd.unique(values),key=lambda value:str(value))
        if len(labels)!=5:
            raise ValueError(f'{kind} must contain exactly five outer folds; found {labels} in {source_col}.')
        out=frame.copy()
        out['_display_fold']=values.map({label:i for i,label in enumerate(labels)}).astype(int)
        return out,'_display_fold',source_col

    for row,task in enumerate(['PTAL','EPC']):
        for col,(title,folds,kind) in enumerate(schemes):
            ax=axes[row,col];add_map_context(ax,add_basemap=True)
            ff=folds[folds.task.astype(str).str.upper().eq(task)].copy() if 'task' in folds else folds.copy()
            if ff.empty: raise ValueError(f'No {task} rows found in the {kind} fold assignment.')
            ff,fold_col,source_fold_col=normalise_fold_assignment(ff,kind)
            if kind=='borough' and borough_code_col and 'borough_code' in ff:
                bmap=ff[['borough_code',fold_col]].drop_duplicates('borough_code')
                bp=boroughs.merge(bmap,left_on=borough_code_col,right_on='borough_code',how='left')
                bp.plot(ax=ax,column=fold_col,cmap=ListedColormap(FOLD_COLORS),categorical=True,edgecolor='white',linewidth=.45,alpha=.82,zorder=2)
                plotted.extend(bmap.assign(task=task,scheme=kind,source_fold_column=source_fold_col).to_dict('records'))
            else:
                gp=points[points.task.eq(task)][['sample_id','geometry']].merge(ff[['sample_id',fold_col]],on='sample_id',how='inner')
                gp=gpd.GeoDataFrame(gp,geometry='geometry',crs=points.crs)
                if gp.empty: raise ValueError(f'No analytical samples matched the {task} {kind} fold assignment.')
                if kind=='random' and len(gp)>5000: gp=gp.sample(5000,random_state=42)
                ax.scatter(gp.geometry.x,gp.geometry.y,c=gp[fold_col],cmap=ListedColormap(FOLD_COLORS),norm=BoundaryNorm(np.arange(-.5,5.5),5),s=2.2 if task=='EPC' else 4,alpha=.72,linewidths=0,rasterized=True,zorder=2)
                plotted.extend(gp.drop(columns='geometry').assign(task=task,scheme=kind,source_fold_column=source_fold_col).to_dict('records'))
            if boroughs is not None: boroughs.boundary.plot(ax=ax,color='#525A5E',lw=.27,alpha=.7,zorder=4)
            if row==0: ax.set_title(title,loc='left')
            if col==0: ax.text(-.08,.5,task,transform=ax.transAxes,rotation=90,ha='center',va='center',fontsize=10,fontweight='bold')
            panel_label(ax,chr(97+row*3+col))
    handles=[patches.Patch(facecolor=c,edgecolor='none',label=f'Fold {i+1}') for i,c in enumerate(FOLD_COLORS)]
    fig.legend(handles=handles,ncol=5,loc='lower center',frameon=False,bbox_to_anchor=(.5,.01))
    fig.tight_layout(rect=(0,.05,1,1),w_pad=.5,h_pad=.7)
    save_figure(fig,'Figure_4','spatial validation design','Methods / RQ3',
                [cfg.RANDOM_OUTER_FOLDS_PATH,cfg.RIDGE_OUTER_FOLDS_PATH,cfg.GEOMETRIC_OUTER_FOLDS_PATH,boundary_source],
                'Fixed outer-fold assignment',
                'Colours identify folds within each task and design; fold numbers have no ordinal meaning.',data=pd.DataFrame(plotted))


## Figure 5. Task-specific image support and sampling parameters

The same location is shown at 300 m and 150 m aerial support. Archived Street View thumbnails demonstrate the different maximum image counts and radii used for PTAL and EPC.


In [ ]:
if BUILD_MAIN_FIGURES:
    fig=plt.figure(figsize=(14.8,8.2));gs=fig.add_gridspec(2,2,height_ratios=[1.05,1.0],wspace=.12,hspace=.28)
    ax300=fig.add_subplot(gs[0,0]);ax150=fig.add_subplot(gs[0,1])
    for ax,image,title,extent,colour,label in [
        (ax300,aerial_300,'Neighbourhood-scale context (PTAL)',300,COLORS['individual'],'300 m × 300 m'),
        (ax150,aerial_150,'Local context (EPC)',150,COLORS['fusion'],'150 m × 150 m')]:
        ax.imshow(image);ax.scatter([image.shape[1]/2],[image.shape[0]/2],s=34,facecolor='white',edgecolor=colour,lw=1.5,zorder=4)
        ax.add_patch(patches.Rectangle((8,8),image.shape[1]-16,image.shape[0]-16,fill=False,edgecolor=colour,lw=2))
        ax.set_axis_off();ax.set_title(title,loc='left');ax.text(.02,.025,label,transform=ax.transAxes,color='white',fontsize=8,fontweight='bold',path_effects=[pe.withStroke(linewidth=2.5,foreground='black')])
    panel_label(ax300,'a');panel_label(ax150,'b')

    # Nested thumbnail grids: 8 within 300 m and 4 within 150 m.
    for col,(images,title,colour,radius,k) in enumerate([(ptal_street,'PTAL Street View: ≤8 images within 300 m',COLORS['individual'],300,8),(epc_street,'EPC Street View: ≤4 images within 150 m',COLORS['fusion'],150,4)]):
        ncols=4; nrows=2 if k==8 else 1
        sub=gs[1,col].subgridspec(nrows,ncols,wspace=.035,hspace=.055)
        axs=[]
        for i,(im,d,member) in enumerate(images):
            ax=fig.add_subplot(sub[i//ncols,i%ncols]);ax.imshow(im);ax.set_axis_off();ax.text(.03,.05,f'{d:.0f} m',transform=ax.transAxes,color='white',fontsize=6.4,path_effects=[pe.withStroke(linewidth=2,foreground='black')]);axs.append(ax)
        axs[0].text(0,1.16,title,transform=axs[0].transAxes,fontsize=10,fontweight='semibold',color=colour,ha='left')
        panel_label(axs[0],'c' if col==0 else 'd')
    fig.text(.5,.005,'Aerial panels share one fixed centre; Street View thumbnails are the nearest readable archived images within each task-specific radius.',ha='center',fontsize=7.1,color=COLORS['mid'])
    support_data=pd.concat([
        pd.DataFrame({'task':'PTAL','source':'Street View','distance_m':[x[1] for x in ptal_street],'radius_m':300}),
        pd.DataFrame({'task':'EPC','source':'Street View','distance_m':[x[1] for x in epc_street],'radius_m':150})],ignore_index=True)
    save_figure(fig,'Figure_5','task specific image support and sampling parameters','Methods',
                [AERIAL_SOURCE_INDEX,STREET_IMAGE_INVENTORY,RAW_STREET_ZIP],
                'Real image context, crop extent, sampling radius and image count',
                'Examples explain methodological support and are not selected by outcomes or model performance.',data=support_data)


## Figure 6. Primary borough-held-out benchmark

Absolute held-out performance is separated from incremental value. Grey points are the five fixed borough folds; coloured means and SD intervals are labelled directly.


In [ ]:
if BUILD_MAIN_FIGURES:
    selected={'PTAL':['PTAL_spatial_baseline','SatCLIP','TESSERA','AlphaEarth','DINOv2','StreetView_CLIP_plus_metadata','Sky_and_space','Street_and_sky','All_representations_plus_SV_metadata'],
              'EPC':['EPC_controls_sparse','EPC_controls_extensive','SatCLIP','TESSERA','AlphaEarth','DINOv2','StreetView_CLIP_plus_metadata','Sky_and_space','All_representations_plus_SV_metadata']}
    category=lambda m:'control' if ('baseline' in m or 'controls' in m) else ('fusion' if m in ['Sky_and_space','Street_and_sky','All_representations_plus_SV_metadata'] else 'individual')
    benchmark_label=lambda m:'Privileged extended property controls' if m=='EPC_controls_extensive' else short_model(m)
    fig,axes=plt.subplots(1,2,figsize=(15.3,7.2));plot_rows=[]
    for ax,task in zip(axes,['PTAL','EPC']):
        sub=core_fold[(core_fold.task==task)&core_fold.feature_set.isin(selected[task])].copy();order=[m for m in selected[task] if m in set(sub.feature_set)]
        for yi,m in enumerate(order):
            g=sub[sub.feature_set==m].sort_values('outer_fold');mean=g.r2.mean();sd=g.r2.std(ddof=1);cat=category(m);colour=COLORS[cat]
            ax.scatter(g.r2,yi+np.linspace(-.08,.08,len(g)),s=16,color='#AAB0B3',alpha=.68,zorder=2)
            ax.errorbar(mean,yi,xerr=sd,fmt='o',ms=6.2,color=colour,ecolor=colour,capsize=3,lw=1.35,zorder=3)
            ax.annotate(f'{mean:.3f}',(mean,yi),xytext=(7,0),textcoords='offset points',va='center',fontsize=7.3,color=colour,fontweight='semibold')
            plot_rows.extend(g.assign(reader_label=benchmark_label(m),category=cat).to_dict('records'))
        ax.set_yticks(np.arange(len(order)),[benchmark_label(m) for m in order]);ax.invert_yaxis();ax.set_xlabel('Mean held-out-fold R²');ax.set_title(task,loc='left');ax.grid(axis='x',color='#E2E3E3',lw=.65);ax.set_axisbelow(True);ax.axvline(0,color='#B8BDC0',lw=.6)
        cats=[category(m) for m in order]
        for i in range(len(cats)-1):
            if cats[i]!=cats[i+1]:ax.axhline(i+.5,color='#D9DCDE',lw=.8)
    handles=[Line2D([],[],marker='o',ls='none',color=COLORS[k],label=l) for k,l in [('control','Controls'),('individual','Single representation'),('fusion','Combination')]]
    fig.legend(handles=handles,ncol=3,frameon=False,loc='lower center',bbox_to_anchor=(.5,.005));panel_label(axes[0],'a');panel_label(axes[1],'b');fig.tight_layout(rect=(0,.06,1,1),w_pad=2.2)
    save_figure(fig,'Figure_6','primary borough held out benchmark','Results RQ1',[cfg.RIDGE_CORE_RESULTS_PATH],
                'Mean ± SD of held-out-fold R²','Absolute predictive performance; means are not pooled OOF R².',data=pd.DataFrame(plot_rows))


## Figure 7. Incremental value beyond matched controls

Paired fold-level ΔR² isolates additional predictive information beyond the matched task controls. The zero reference is deliberately light and dashed.


In [ ]:
if BUILD_MAIN_FIGURES:
    delta_models={'PTAL':['PTAL_spatial_baseline__plus__DINOv2','PTAL_spatial_baseline__plus__StreetView_CLIP_plus_metadata','PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata'],
                  'EPC':['EPC_controls_sparse__plus__DINOv2','EPC_controls_sparse__plus__Sky_and_space','EPC_controls_sparse__plus__All_representations_plus_SV_metadata','EPC_controls_extensive__plus__TESSERA','EPC_controls_extensive__plus__All_representations_plus_SV_metadata']}
    labels={'PTAL_spatial_baseline__plus__DINOv2':'Aerial image (DINOv2)','PTAL_spatial_baseline__plus__StreetView_CLIP_plus_metadata':'Street view (CLIP) + coverage','PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata':'Full combination','EPC_controls_sparse__plus__DINOv2':'Aerial image (compact controls)','EPC_controls_sparse__plus__Sky_and_space':'Aerial + satellite (compact)','EPC_controls_sparse__plus__All_representations_plus_SV_metadata':'Full combination (compact)','EPC_controls_extensive__plus__TESSERA':'TESSERA (privileged extended controls)','EPC_controls_extensive__plus__All_representations_plus_SV_metadata':'Full combination (privileged extended)'}
    fig,axes=plt.subplots(1,2,figsize=(13.8,5.5),gridspec_kw={'width_ratios':[1,1.35]});plotted=[]
    for ax,task in zip(axes,['PTAL','EPC']):
        sub=paired_delta[(paired_delta.task==task)&paired_delta.model_id.isin(delta_models[task])].copy();order=sub.groupby('model_id').delta_r2.mean().sort_values().index.tolist()
        for yi,m in enumerate(order):
            g=sub[sub.model_id==m].sort_values('outer_fold');mean=g.delta_r2.mean();sd=g.delta_r2.std(ddof=1);colour=colour_for_delta(mean)
            ax.scatter(g.delta_r2,yi+np.linspace(-.07,.07,len(g)),s=17,color='#A8AEB1',alpha=.78,zorder=2);ax.errorbar(mean,yi,xerr=sd,fmt='o',ms=6.2,color=colour,ecolor=colour,capsize=3,zorder=3);ax.annotate(f'{mean:+.3f}',(mean,yi),xytext=(7,0),textcoords='offset points',va='center',fontsize=7.3,color=colour)
            plotted.extend(g.assign(reader_label=labels[m]).to_dict('records'))
        ax.set_yticks(np.arange(len(order)),[labels[m] for m in order]);ax.axvline(0,color='#C8CCCE',lw=.8,ls='--');ax.set_xlabel('Paired change in held-out R²');ax.set_title(task,loc='left');ax.grid(axis='x',color='#E4E5E5',lw=.65);ax.set_axisbelow(True)
    panel_label(axes[0],'a');panel_label(axes[1],'b');fig.tight_layout(w_pad=2.2)
    save_figure(fig,'Figure_7','incremental value beyond matched controls','Results RQ2',[cfg.INCREMENTAL_DELTAS_PATH],
                'Fold-level paired ΔR²; mean ± SD','Predictive information beyond matched controls; ΔR² is not a causal effect.',data=pd.DataFrame(plotted))


## Figure 8. Why EPC extended controls are strong

The control ablation is separated from representation gains. A horizontal bar chart shows how much construction age, floor area and their combination add to compact EPC controls.


In [ ]:
if BUILD_MAIN_FIGURES:
    abl=control_ablation.copy();name_col='added_feature_set' if 'added_feature_set' in abl else 'model_id'
    label_map={'floor_area':'Floor area only','construction_age':'Construction age only','floor_area_plus_construction_age':'Construction age + floor area'}
    abl['label']=abl[name_col].astype(str).map(label_map).fillna(abl[name_col].astype(str).str.replace('_',' ',regex=False));abl=abl.sort_values('mean_delta_r2')
    fig,ax=plt.subplots(figsize=(9.7,4.3));y=np.arange(len(abl));cols=[colour_for_delta(v) for v in abl.mean_delta_r2]
    ax.barh(y,abl.mean_delta_r2,color=cols,alpha=.82,height=.54,xerr=abl.sd_delta_r2,error_kw=dict(ecolor='#596064',capsize=3,lw=.9))
    for yi,v in enumerate(abl.mean_delta_r2): ax.text(v+.008,yi,f'{v:+.4f}',va='center',fontsize=8,fontweight='semibold',color=cols[yi])
    ax.set_yticks(y,abl.label);ax.set_xlabel('Change in held-out R² beyond compact EPC controls');ax.axvline(0,color='#C8CCCE',lw=.8,ls='--');ax.grid(axis='x',color='#E3E4E4',lw=.65);ax.set_axisbelow(True);fig.tight_layout()
    save_figure(fig,'Figure_8','why EPC extended controls are strong','Results RQ2',[cfg.INCREMENTAL_ABLATION_SUMMARY_PATH],
                'Mean paired ΔR² ± SD across borough folds','Construction-age information dominates this frozen control ablation; predictive, not causal.',data=abl)


## Figure 9. Spatial generalisation across evaluation designs

Categorical slope plots show performance change from random folds to held-out borough-level groups and continuous regions. Only controls, DINOv2 and the full combination are retained to avoid a line cloud.


In [ ]:
if BUILD_MAIN_FIGURES:
    scheme_order=['random_nested_5fold','borough_grouped_nested_5fold','geometric_region_nested_5fold'];x=np.arange(3);scheme_labels=['Random','Borough-level','Continuous region']
    reps={'PTAL':[('PTAL_spatial_baseline','Controls',COLORS['control']),('PTAL_spatial_baseline__plus__DINOv2','+ DINOv2',COLORS['individual']),('PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata','+ full combination',COLORS['fusion'])],
          'EPC':[('EPC_controls_sparse','Compact controls',COLORS['control']),('EPC_controls_sparse__plus__DINOv2','+ DINOv2',COLORS['individual']),('EPC_controls_sparse__plus__All_representations_plus_SV_metadata','+ full combination',COLORS['fusion'])]}
    fig,axes=plt.subplots(1,2,figsize=(12.9,5.6),sharex=True);plotted=[]
    for ax,task in zip(axes,['PTAL','EPC']):
        for m,label,colour in reps[task]:
            g=protocol_summary[(protocol_summary.task==task)&(protocol_summary.model_id==m)].set_index('validation_scheme');vals=np.array([g.loc[s,'pooled_r2'] for s in scheme_order])
            ax.plot(x,vals,marker='o',ms=6,lw=1.7,color=colour,label=label);ax.text(2.04,vals[-1],f'{vals[-1]:.3f}\n({vals[-1]-vals[0]:+.3f})',va='center',fontsize=7,color=colour)
            plotted.extend([{'task':task,'model_id':m,'label':label,'validation_scheme':s,'pooled_r2':v} for s,v in zip(scheme_order,vals)])
        ax.set_xticks(x,scheme_labels);ax.set_ylabel('Pooled held-out OOF R²');ax.set_title(task,loc='left');ax.grid(axis='y',color='#E1E3E3',lw=.65);ax.set_axisbelow(True)
    handles=[Line2D([],[],marker='o',lw=1.7,color=c,label=l) for _,l,c in reps['PTAL']]
    fig.legend(handles=handles,ncol=3,frameon=False,loc='upper center',bbox_to_anchor=(.5,.995))
    panel_label(axes[0],'a');panel_label(axes[1],'b');fig.tight_layout(rect=(0,0,1,.92),w_pad=2.4)
    save_figure(fig,'Figure_9','spatial generalisation across evaluation designs','Results RQ3',[cfg.GEOMETRIC_PROTOCOL_SUMMARY_PATH],
                'Pooled OOF R² across three validation designs','Lines link the same specification across different deployment questions; folds and test sets are not paired.',data=pd.DataFrame(plotted))


## Figure 10. Observed versus held-out predictions and calibration

Small grey OOF pairs are overlaid with smoothed density contours, a 45-degree ideal line and a descriptive calibration line. Only pooled OOF R² and calibration slope are annotated.


In [ ]:
if BUILD_MAIN_FIGURES:
    inc_pred=pd.read_parquet(cfg.INCREMENTAL_PREDICTIONS_PATH)
    for c in ['sample_id','task','model_id']: inc_pred[c]=inc_pred[c].astype(str)
    inc_pred=numeric(inc_pred,['y_true','y_pred','outer_fold'])
    panels=[('PTAL','PTAL_spatial_baseline','Spatial controls'),('PTAL','PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata','Full combination'),('EPC','EPC_controls_sparse','Compact controls'),('EPC','EPC_controls_sparse__plus__All_representations_plus_SV_metadata','Full combination')]
    fig,axes=plt.subplots(2,2,figsize=(11.7,9.6));plotted=[];task_limits={}
    for task in ['PTAL','EPC']:
        s=inc_pred[(inc_pred.task==task)&inc_pred.model_id.isin([m for t,m,_ in panels if t==task])];lo=np.nanpercentile(np.r_[s.y_true,s.y_pred],.5);hi=np.nanpercentile(np.r_[s.y_true,s.y_pred],99.5);pad=.04*(hi-lo);task_limits[task]=(lo-pad,hi+pad)
    for i,(ax,(task,m,label)) in enumerate(zip(axes.ravel(),panels)):
        g=inc_pred[(inc_pred.task==task)&(inc_pred.model_id==m)].dropna(subset=['y_true','y_pred']).copy();lo,hi=task_limits[task]
        pts=g.sample(min(4500,len(g)),random_state=42);ax.scatter(pts.y_true,pts.y_pred,s=4,color='#777D81',alpha=.10,linewidths=0,rasterized=True)
        H,xe,ye=np.histogram2d(g.y_true,g.y_pred,bins=58,range=[[lo,hi],[lo,hi]]);H=gaussian_filter(H,1.2);xc=(xe[:-1]+xe[1:])/2;yc=(ye[:-1]+ye[1:])/2
        positive=H[H>0];levels=np.quantile(positive,[.72,.86,.94,.98]) if len(positive) else []
        if len(np.unique(levels))>1: ax.contour(xc,yc,H.T,levels=np.unique(levels),colors=COLORS['individual'],linewidths=[.7,.9,1.1,1.3])
        ax.plot([lo,hi],[lo,hi],ls='--',lw=.9,color='#8A9094');slope,intercept=np.polyfit(g.y_true,g.y_pred,1);xs=np.linspace(lo,hi,100);ax.plot(xs,slope*xs+intercept,color=COLORS['fusion'],lw=1.5)
        score=r2_score(g.y_true,g.y_pred);ax.text(.04,.95,f'Pooled OOF R² = {score:.3f}\nCalibration slope = {slope:.2f}',transform=ax.transAxes,va='top',fontsize=8.2,bbox=dict(facecolor='white',edgecolor='#D1D4D5',alpha=.94,pad=4))
        ax.set(xlim=(lo,hi),ylim=(lo,hi),xlabel='Observed value',ylabel='Held-out OOF prediction',title=f'{task}: {label}');panel_label(ax,chr(97+i));plotted.extend(g[['sample_id','task','model_id','outer_fold','y_true','y_pred']].to_dict('records'))
    fig.tight_layout(w_pad=1.6,h_pad=1.6)
    save_figure(fig,'Figure_10','observed versus held out predictions and calibration','Results RQ1–RQ2',[cfg.INCREMENTAL_PREDICTIONS_PATH],
                'OOF pairs, density contour, pooled R² and calibration slope','Held-out descriptive diagnostics; no in-sample fitted values or recalibration.',data=pd.DataFrame(plotted))


## Figure 11. Prediction error across the target range

Existing borough-held-out OOF predictions are grouped into observed-target deciles. MAE shows error magnitude; signed error shows systematic under- or over-prediction. This directly tests regression-to-the-mean behaviour.


In [ ]:
if BUILD_MAIN_FIGURES:
    compare={'PTAL':[('PTAL_spatial_baseline','Controls'),('PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata','Full combination')],
             'EPC':[('EPC_controls_sparse','Compact controls'),('EPC_controls_sparse__plus__All_representations_plus_SV_metadata','Full combination')]}
    rows=[]
    for task,models in compare.items():
        ref=inc_pred[(inc_pred.task==task)&(inc_pred.model_id==models[0][0])][['sample_id','y_true']].drop_duplicates('sample_id').copy();ref['target_quantile_group']=pd.qcut(ref.y_true,10,labels=False,duplicates='drop')+1
        for m,label in models:
            g=inc_pred[(inc_pred.task==task)&(inc_pred.model_id==m)].merge(ref[['sample_id','target_quantile_group']],on='sample_id',how='inner');g['error']=g.y_pred-g.y_true
            agg=g.groupby('target_quantile_group',as_index=False).agg(mae=('error',lambda s:s.abs().mean()),bias=('error','mean'),n=('sample_id','size'),mean_target=('y_true','mean'));agg['task']=task;agg['model_id']=m;agg['label']=label;rows.extend(agg.to_dict('records'))
    error_by_decile=pd.DataFrame(rows)
    fig,axes=plt.subplots(2,2,figsize=(12.6,8.2),sharex='col')
    for col,task in enumerate(['PTAL','EPC']):
        for label,colour in [('Controls' if task=='PTAL' else 'Compact controls',COLORS['control']),('Full combination',COLORS['fusion'])]:
            g=error_by_decile[(error_by_decile.task==task)&(error_by_decile.label==label)]
            axes[0,col].plot(g.target_quantile_group,g.mae,marker='o',lw=1.6,color=colour,label=label);axes[1,col].plot(g.target_quantile_group,g.bias,marker='o',lw=1.6,color=colour,label=label)
        n_groups=int(error_by_decile.loc[error_by_decile.task==task,'target_quantile_group'].nunique())
        axes[0,col].set_title(f'{task} ({n_groups} groups)',loc='left');axes[0,col].set_ylabel('Mean absolute error');axes[1,col].set_ylabel('Mean signed error\n(predicted − observed)');axes[1,col].set_xlabel('Observed-target quantile group');axes[1,col].axhline(0,color='#B8BDC0',lw=.8,ls='--');axes[0,col].legend(frameon=False)
        for ax in axes[:,col]: ax.grid(axis='y',color='#E2E4E4',lw=.65);ax.set_axisbelow(True);ax.set_xticks(range(1,n_groups+1));ax.set_xlim(.6,n_groups+.4)
    for i,ax in enumerate(axes.ravel()): panel_label(ax,chr(97+i))
    fig.tight_layout(w_pad=2,h_pad=1.4)
    save_figure(fig,'Figure_11','prediction error across the target range','Results RQ1–RQ2',[cfg.INCREMENTAL_PREDICTIONS_PATH],
                'OOF MAE and signed error by observed-target quantile group','Post-hoc descriptive aggregation; tied PTAL values yield nine groups rather than ten and are not split arbitrarily.',data=error_by_decile)


## Figure 12. Residual geography under continuous-region validation

Residual hexagons are retained only here because spatial aggregation serves the scientific question. Moran's I remains calculated from original OOF residuals, not from the display hexagons.


In [ ]:
if BUILD_MAIN_FIGURES:
    geo_pred=pd.read_parquet(cfg.GEOMETRIC_CV_PREDICTIONS_PATH)
    for c in ['sample_id','task','model_id']: geo_pred[c]=geo_pred[c].astype(str)
    geo_pred=numeric(geo_pred,['y_true','y_pred']);geo_pred=geo_pred.merge(points[['sample_id','geometry']],on='sample_id',how='left',validate='many_to_one')
    # pandas.merge drops the GeoDataFrame subclass even though the geometry
    # column survives. Restore it explicitly before accessing geometry.x/y.
    geo_pred=gpd.GeoDataFrame(geo_pred,geometry='geometry',crs=points.crs);geo_pred['residual']=geo_pred.y_true-geo_pred.y_pred
    panels=[('PTAL','PTAL_spatial_baseline','Spatial controls'),('PTAL','PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata','Full combination'),('EPC','EPC_controls_sparse','Compact controls'),('EPC','EPC_controls_sparse__plus__All_representations_plus_SV_metadata','Full combination')]
    fig,axes=plt.subplots(2,2,figsize=(12.2,10.0));plotted=[];task_vmax={}
    for task in ['PTAL','EPC']:
        ids=[m for t,m,_ in panels if t==task];task_vmax[task]=float(np.nanpercentile(geo_pred[(geo_pred.task==task)&geo_pred.model_id.isin(ids)].residual.abs(),97.5))
    for i,(ax,(task,m,label)) in enumerate(zip(axes.ravel(),panels)):
        add_map_context(ax,add_basemap=True);g=gpd.GeoDataFrame(geo_pred[(geo_pred.task==task)&(geo_pred.model_id==m)].dropna(subset=['geometry','residual']).copy(),geometry='geometry',crs=points.crs);vmax=task_vmax[task]
        hb=ax.hexbin(g.geometry.x,g.geometry.y,C=g.residual,gridsize=56,reduce_C_function=np.mean,mincnt=1,cmap='RdBu_r',vmin=-vmax,vmax=vmax,linewidths=0,alpha=.88,zorder=2)
        boroughs.boundary.plot(ax=ax,color='#4F565A',lw=.34,alpha=.8,zorder=4);boroughs.dissolve().boundary.plot(ax=ax,color='#22272A',lw=.82,zorder=5)
        lookup=moran_summary[(moran_summary.task==task)&(moran_summary.model_id==m)&(moran_summary.validation_scheme=='geometric_region_nested_5fold')];moran=float(lookup.moran_i.iloc[0]) if len(lookup) else np.nan
        ax.set_title(f'{task}: {label}',loc='left');ax.text(.02,.975,f"Moran's I = {moran:.3f}",transform=ax.transAxes,ha='left',va='top',fontsize=7.7,bbox=dict(facecolor='white',edgecolor='#D0D3D5',pad=3,alpha=.9),zorder=10);panel_label(ax,chr(97+i));plotted.extend(g[['sample_id','task','model_id','y_true','y_pred','residual']].to_dict('records'))
    for row,task in enumerate(['PTAL','EPC']):
        sm=mpl.cm.ScalarMappable(norm=TwoSlopeNorm(vmin=-task_vmax[task],vcenter=0,vmax=task_vmax[task]),cmap='RdBu_r');cbar=fig.colorbar(sm,ax=axes[row,:],fraction=.025,pad=.018);cbar.set_label('Mean residual in display hexagon (observed − predicted)',fontsize=7.8)
    add_north_scale(axes[0,0],10);fig.text(.01,.008,'Basemap: CARTO / OpenStreetMap contributors. Hexagons are display aggregation only.',fontsize=6.3,color=COLORS['mid']);fig.subplots_adjust(wspace=.05,hspace=.11,right=.91,bottom=.035)
    save_figure(fig,'Figure_12','residual geography under continuous region validation','Results RQ3',[cfg.GEOMETRIC_CV_PREDICTIONS_PATH,cfg.GEOMETRIC_RESIDUAL_MORAN_PATH,cfg.FINAL_MODEL_TABLE_PATH],
                "Hexagon-mean OOF residual; Moran's I on original residuals","Representations reduce residual clustering but do not identify a specific omitted cause.",data=pd.DataFrame(plotted))


## Figure 13. Where representations improve prediction geographically

For each borough, baseline and full-model borough-held-out OOF MAE are compared. Positive ΔMAE means the full representation combination reduces error. This is a descriptive geography of improvement, not a new model.


In [ ]:
if BUILD_MAIN_FIGURES:
    compare={'PTAL':('PTAL_spatial_baseline','PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata'),'EPC':('EPC_controls_sparse','EPC_controls_sparse__plus__All_representations_plus_SV_metadata')}
    rows=[]
    for task,(base,full) in compare.items():
        b=inc_pred[(inc_pred.task==task)&(inc_pred.model_id==base)][['sample_id','y_true','y_pred']].rename(columns={'y_pred':'base_pred'});f=inc_pred[(inc_pred.task==task)&(inc_pred.model_id==full)][['sample_id','y_pred']].rename(columns={'y_pred':'full_pred'})
        g=b.merge(f,on='sample_id',validate='one_to_one').merge(map_df[['sample_id','borough_code']],on='sample_id',validate='many_to_one');g['base_abs_error']=(g.y_true-g.base_pred).abs();g['full_abs_error']=(g.y_true-g.full_pred).abs()
        agg=g.groupby('borough_code',as_index=False).agg(n=('sample_id','size'),baseline_mae=('base_abs_error','mean'),full_mae=('full_abs_error','mean'));agg['delta_mae']=agg.baseline_mae-agg.full_mae;agg['task']=task;rows.extend(agg.to_dict('records'))
    borough_improvement=pd.DataFrame(rows)
    fig,axes=plt.subplots(1,2,figsize=(12.8,6.0))
    for ax,task in zip(axes,['PTAL','EPC']):
        g=borough_improvement[borough_improvement.task==task];limit=float(np.nanmax(np.abs(g.delta_mae)));norm=TwoSlopeNorm(vmin=-limit,vcenter=0,vmax=limit)
        bp=boroughs.merge(g,left_on=borough_code_col,right_on='borough_code',how='left');bp.plot(ax=ax,column='delta_mae',cmap='BrBG',norm=norm,edgecolor='white',linewidth=.5,zorder=2);boroughs.dissolve().boundary.plot(ax=ax,color='#22272A',lw=.85,zorder=4);ax.set_axis_off();ax.set_aspect('equal');positive=int((g.delta_mae>0).sum());ax.set_title(f'{task}: lower error in {positive}/{len(g)} local-authority areas',loc='left');panel_label(ax,'a' if task=='PTAL' else 'b')
        sm=mpl.cm.ScalarMappable(norm=norm,cmap='BrBG');cbar=fig.colorbar(sm,ax=ax,fraction=.038,pad=.012);cbar.set_label(f'{task} MAE reduction\n(baseline − full)',fontsize=7.5)
    fig.subplots_adjust(wspace=.18)
    save_figure(fig,'Figure_13','where representations improve prediction geographically','Results RQ2–RQ3',[cfg.INCREMENTAL_PREDICTIONS_PATH,cfg.FINAL_MODEL_TABLE_PATH,boundary_source],
                'Local-authority-area difference in OOF MAE','Post-hoc descriptive aggregation; area differences are not causal local treatment effects.',data=borough_improvement)


## Figure 14. Model-class robustness

The upper row compares Ridge and XGBoost using identical features and folds. The lower row shows GATv2 minus MLP for the frozen neighbourhood-model comparison. Both panels concern model class rather than new representation inputs.


In [ ]:
if BUILD_MAIN_FIGURES:
    fig,axes=plt.subplots(2,2,figsize=(14.2,10.0));plotted=[]
    model_display={'PTAL_spatial_baseline':'Controls','PTAL_spatial_baseline__plus__DINOv2':'+ DINOv2','PTAL_spatial_baseline__plus__StreetView_CLIP_plus_metadata':'+ street view','PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata':'+ full combination','EPC_controls_sparse':'Compact controls','EPC_controls_sparse__plus__DINOv2':'+ DINOv2','EPC_controls_sparse__plus__All_representations_plus_SV_metadata':'+ full combination','EPC_controls_extensive':'Privileged extended controls','EPC_controls_extensive__plus__TESSERA':'+ TESSERA','EPC_controls_extensive__plus__All_representations_plus_SV_metadata':'+ full combination (privileged extended)'}
    for col,task in enumerate(['PTAL','EPC']):
        ax=axes[0,col];sub=xgb_vs_ridge[xgb_vs_ridge.task==task].copy().sort_values('ridge_pooled_r2');y=np.arange(len(sub))
        for i,(_,r) in enumerate(sub.iterrows()):
            ax.plot([r.ridge_pooled_r2,r.xgboost_pooled_r2],[i,i],color='#C9CDCF',lw=1.4);ax.text(r.ridge_pooled_r2,i+.18,f'{r.ridge_pooled_r2:.3f}',ha='center',fontsize=6.7,color=COLORS['ridge']);ax.text(r.xgboost_pooled_r2,i-.22,f'{r.xgboost_pooled_r2:.3f} ({r.xgboost_pooled_r2-r.ridge_pooled_r2:+.3f})',ha='center',fontsize=6.7,color=COLORS['xgboost'])
        ax.scatter(sub.ridge_pooled_r2,y,s=38,color=COLORS['ridge'],label='Ridge',zorder=2);ax.scatter(sub.xgboost_pooled_r2,y,s=38,color=COLORS['xgboost'],label='XGBoost',zorder=3);ax.set_yticks(y,[model_display.get(m,short_model(m)) for m in sub.model_id]);ax.set_xlabel('Pooled borough-level-held-out OOF R²');ax.set_title(f'{task}: Ridge → XGBoost',loc='left');ax.grid(axis='x',color='#E1E3E3',lw=.65);ax.margins(x=.09,y=.18);plotted.extend(sub.assign(panel='Ridge_vs_XGBoost').to_dict('records'))
        ax2=axes[1,col];g=gat_vs_mlp[gat_vs_mlp.task==task].copy().sort_values('mean_delta_r2_gatv2_minus_mlp');yy=np.arange(len(g));ax2.errorbar(g.mean_delta_r2_gatv2_minus_mlp,yy,xerr=g.sd_delta_r2,fmt='o',ms=5.8,color=COLORS['gat'],ecolor=COLORS['gat'],capsize=3);ax2.axvline(0,color='#C8CCCE',lw=.8,ls='--');ax2.set_yticks(yy,[model_display.get(m,short_model(m)) for m in g.model_id]);ax2.set_xlabel('Mean paired ΔR²: GATv2 − MLP');ax2.set_title(f'{task}: neighbourhood model class',loc='left');ax2.grid(axis='x',color='#E1E3E3',lw=.65);ax2.margins(x=.1,y=.18);plotted.extend(g.assign(panel='GATv2_vs_MLP').to_dict('records'))
    for i,ax in enumerate(axes.ravel()): panel_label(ax,chr(97+i));ax.set_axisbelow(True)
    handles=[Line2D([],[],marker='o',ls='none',color=COLORS['ridge'],label='Ridge'),Line2D([],[],marker='o',ls='none',color=COLORS['xgboost'],label='XGBoost')]
    fig.legend(handles=handles,ncol=2,frameon=False,loc='upper center',bbox_to_anchor=(.5,.995))
    fig.tight_layout(rect=(0,0,1,.95),w_pad=2,h_pad=2)
    save_figure(fig,'Figure_14','model class robustness','Results robustness',[cfg.XGBOOST_VS_RIDGE_SUMMARY_PATH,cfg.GATV2_VS_MLP_SUMMARY_PATH],
                'Pooled OOF R² and paired model-class ΔR²','Model-class sensitivity with frozen inputs and folds; not an additional representation search.',data=pd.DataFrame(plotted))


## Figure 15. EPC target, reliability and timing robustness

Alternative EPC outcome definitions are separated from prediction error across record-year bands. The two panels now share one coherent purpose: bounding dependence on EPC data construction and timing.


In [ ]:
if BUILD_MAIN_FIGURES:
    fig,axes=plt.subplots(1,2,figsize=(14.0,5.8));plotted=[]
    proto_order=['n3_main_target','uprn_covered_main_target','uprn_only_target','record_year_adjusted'];proto_labels=['Primary: ≥3 properties','UPRN-covered','UPRN-only','Year-adjusted']
    selected=['EPC_controls_sparse__plus__DINOv2','EPC_controls_sparse__plus__All_representations_plus_SV_metadata','EPC_controls_extensive__plus__TESSERA'];colours=[COLORS['individual'],COLORS['fusion'],COLORS['control']]
    robustness_label={'EPC_controls_sparse':'Compact controls','EPC_controls_sparse__plus__DINOv2':'Compact controls + aerial-image embedding (DINOv2)','EPC_controls_sparse__plus__All_representations_plus_SV_metadata':'Compact controls + full combination','EPC_controls_extensive':'Privileged extended property controls','EPC_controls_extensive__plus__TESSERA':'Privileged extended controls + satellite time-series embedding (TESSERA)'}
    for colour,m in zip(colours,selected):
        g=epc_protocols[epc_protocols.model_id==m].set_index('protocol_id').reindex(proto_order);label=robustness_label[m];axes[0].plot(range(4),g.mean_delta_r2,marker='o',lw=1.55,color=colour,label=label);axes[0].text(3.05,g.mean_delta_r2.iloc[-1],label,fontsize=6.6,color=colour,va='center');plotted.extend(g.reset_index().assign(panel='target_protocol',model_id=m).to_dict('records'))
    axes[0].axhline(0,color='#C8CCCE',lw=.8,ls='--');axes[0].set_xticks(range(4),proto_labels,rotation=16,ha='right');axes[0].set_xlim(-.15,4.2);axes[0].set_ylabel('Mean paired ΔR²');axes[0].set_title('Outcome-definition sensitivity',loc='left')
    selected_year=['EPC_controls_sparse','EPC_controls_sparse__plus__All_representations_plus_SV_metadata','EPC_controls_extensive']
    for colour,m in zip(colours,selected_year):
        g=epc_year_bins[epc_year_bins.model_id==m].sort_values('mean_record_year');label=robustness_label[m];axes[1].plot(g.mean_record_year,g.mean_absolute_error,marker='o',lw=1.55,color=colour,label=label);axes[1].text(g.mean_record_year.iloc[-1]+.08,g.mean_absolute_error.iloc[-1],label,fontsize=6.6,color=colour,va='center');plotted.extend(g.assign(panel='record_year',model_id=m).to_dict('records'))
    axes[1].set_xlabel('Mean EPC record year within band');axes[1].set_ylabel('Mean absolute error');axes[1].set_title('Prediction error by record timing',loc='left');axes[1].margins(x=.14)
    for i,ax in enumerate(axes):panel_label(ax,chr(97+i));ax.grid(axis='y',color='#E1E3E3',lw=.65);ax.set_axisbelow(True)
    fig.tight_layout(w_pad=2)
    save_figure(fig,'Figure_15','EPC target reliability and timing robustness','Results robustness',[cfg.EPC_ROBUST_INCREMENTAL_SUMMARY_PATH,cfg.EPC_TEMPORAL_BIN_PATH],
                'Paired ΔR² and MAE by EPC protocol/timing','Robustness bounds for EPC data construction; not a new target-selection round.',data=pd.DataFrame(plotted))


## Appendix figures

Dimension sensitivity, targeted DINOv2+SatCLIP complementarity and Street View coverage remain traceable but do not compete with the fifteen main-text questions.


In [ ]:
if BUILD_APPENDIX_FIGURES:
    # A1: PCA64 dimension sensitivity.
    a=pca_comparison.copy();a['label']=a.task+': '+a.feature_set.map(short_model);a=a.sort_values('mean_delta_r2')
    fig,ax=plt.subplots(figsize=(10.8,max(4.5,.38*len(a)+1.5)));y=np.arange(len(a));ax.errorbar(a.mean_delta_r2,y,xerr=a.sd_delta_r2,fmt='o',ms=5.3,color=COLORS['individual'],ecolor=COLORS['individual'],capsize=3);ax.axvline(0,color='#C8CCCE',lw=.8,ls='--');ax.set_yticks(y,a.label);ax.set_xlabel('Mean paired ΔR²: PCA-64 minus native dimensions');ax.grid(axis='x',color='#E1E3E3',lw=.65);fig.tight_layout()
    save_figure(fig,'Appendix_A1','PCA64 representation dimension sensitivity','Appendix',[cfg.PCA64_COMPARISON_SUMMARY_PATH],'Mean paired ΔR² ± SD','Dimension sensitivity only; not a new model-selection round.',placement='Appendix',data=a)

    # A2: targeted SatCLIP added to DINOv2.
    comp=dino_satclip[dino_satclip.comparator_role.astype(str).eq('DINOv2_only')].copy()
    def satclip_context_label(task,model_id):
        model_id=str(model_id)
        if 'spatial_baseline' in model_id: return 'PTAL — spatial controls'
        if 'controls_sparse' in model_id: return 'EPC — compact controls'
        if 'controls_extensive' in model_id: return 'EPC — privileged extended controls'
        return f'{task} — representation only'
    comp['label']=[satclip_context_label(t,m) for t,m in zip(comp.task.astype(str),comp.new_model_id.astype(str))];comp=comp.sort_values('mean_delta_r2')
    fig,ax=plt.subplots(figsize=(9.8,max(3.5,.52*len(comp)+1.4)));y=np.arange(len(comp));ax.errorbar(comp.mean_delta_r2,y,xerr=comp.sd_delta_r2,fmt='o',ms=5.5,color=COLORS['individual'],ecolor=COLORS['individual'],capsize=3);ax.axvline(0,color='#C8CCCE',lw=.8,ls='--');ax.set_yticks(y,comp.label);ax.set_xlabel('Paired ΔR²: DINOv2 + SatCLIP minus DINOv2');ax.grid(axis='x',color='#E1E3E3',lw=.65);fig.tight_layout()
    save_figure(fig,'Appendix_A2','SatCLIP added to DINOv2','Appendix',[DINO_SATCLIP_COMPARISON],'Mean paired ΔR² ± SD','Near-zero change does not prove encoder equivalence.',placement='Appendix',data=comp)

    # A3: Street View coverage by local-authority area.
    if 'sv_has_streetview' in points:
        from matplotlib.ticker import PercentFormatter
        coverage=points.dropna(subset=['sv_has_streetview']).groupby(['task','borough_code'],as_index=False).agg(coverage=('sv_has_streetview','mean'),n=('sample_id','size'));coverage['borough_name']=coverage.borough_code.map(cfg.LONDON_BOROUGH_CODE_TO_NAME).fillna(coverage.borough_code);order=coverage.groupby('borough_name').coverage.mean().sort_values().index.tolist();xmin=max(0,float(coverage.coverage.min())-.05)
        fig,axes=plt.subplots(1,2,figsize=(12.8,8.2),sharey=True)
        for ax,task,colour in zip(axes,['PTAL','EPC'],[COLORS['individual'],COLORS['fusion']]):
            g=coverage[coverage.task==task].set_index('borough_name').reindex(order);ax.scatter(g.coverage,np.arange(len(order)),s=24,color=colour);ax.set_yticks(np.arange(len(order)),order);ax.set_xlabel('Share with street-level imagery');ax.set_title(task,loc='left');ax.set_xlim(xmin,1.005);ax.xaxis.set_major_formatter(PercentFormatter(1));ax.grid(axis='x',color='#E1E3E3',lw=.65)
        panel_label(axes[0],'a');panel_label(axes[1],'b');fig.tight_layout(w_pad=1.4)
        save_figure(fig,'Appendix_A3','street view coverage by local authority area','Appendix',[cfg.FINAL_MODEL_TABLE_PATH],'Local-authority-area share with Street View','Coverage is not image quality or a model-performance result.',placement='Appendix',data=coverage)


## Dissertation tables

The five concise tables retain provenance, representation specification, sample descriptives, headline benchmark results and robustness synthesis. Full fold-level outputs remain supplementary files.


In [ ]:
if BUILD_TABLES:
    from IPython.display import display as ipy_display
    table1=pd.read_csv(cfg.PROVENANCE_TABLE_PATH);table1.to_csv(TABLE_DIR/'Table_1_data_sources_and_provenance.csv',index=False)
    table2=pd.DataFrame([
        ['Location embedding (SatCLIP)','256','Point coordinate','Both'],['Satellite time-series embedding (TESSERA)','128','Sample location','Both'],['Annual EO/environment embedding (AlphaEarth)','64','Sample location','Both'],['Aerial-image embedding (DINOv2)','768','300 m PTAL; 150 m EPC','Both'],['Street-view embedding (CLIP)','512','≤8/300 m PTAL; ≤4/150 m EPC','Both'],['Spatial controls','2','Point coordinate','PTAL'],['Compact property controls','8','EPC postcode neighbourhood','EPC'],['Privileged extended property controls','10','Compact + floor area + construction age, all derived within EPC','EPC']],columns=['Input','Dimensions_or_variables','Spatial_support','Task']);table2.to_csv(TABLE_DIR/'Table_2_representation_and_control_specification.csv',index=False)
    table3=map_df.groupby('task').agg(n_samples=('sample_id','size'),n_local_authority_areas=('borough_code','nunique'),target_mean=('target','mean'),target_sd=('target','std'),target_min=('target','min'),target_max=('target','max')).reset_index();table3.to_csv(TABLE_DIR/'Table_3_sample_and_target_descriptives.csv',index=False)
    headline=['PTAL_spatial_baseline','PTAL_spatial_baseline__plus__DINOv2','PTAL_spatial_baseline__plus__All_representations_plus_SV_metadata','EPC_controls_sparse','EPC_controls_sparse__plus__DINOv2','EPC_controls_sparse__plus__All_representations_plus_SV_metadata','EPC_controls_extensive','EPC_controls_extensive__plus__TESSERA','EPC_controls_extensive__plus__All_representations_plus_SV_metadata'];summary=pd.read_csv(cfg.INCREMENTAL_SUMMARY_PATH);keep=[c for c in ['task','model_id','n_features','mean_r2','sd_r2','pooled_r2','pooled_rmse','pooled_mae'] if c in summary];table4=summary[summary.model_id.isin(headline)][keep].copy();table4.insert(2,'reader_label',table4.model_id.map(short_model));table4.to_csv(TABLE_DIR/'Table_4_headline_borough_cv_results.csv',index=False)
    table5=pd.DataFrame([{'robustness':'Spatial validation','source':str(cfg.GEOMETRIC_PROTOCOL_SUMMARY_PATH),'interpretation':'Performance depends on deployment geography.'},{'robustness':'Non-linear model class','source':str(cfg.XGBOOST_VS_RIDGE_SUMMARY_PATH),'interpretation':'XGBoost changes absolute performance with frozen inputs.'},{'robustness':'Neighbourhood model class','source':str(cfg.GATV2_VS_MLP_SUMMARY_PATH),'interpretation':'GATv2 is compared with the matched MLP.'},{'robustness':'Representation dimension','source':str(cfg.PCA64_COMPARISON_SUMMARY_PATH),'interpretation':'PCA64 bounds unequal embedding dimensions.'},{'robustness':'EPC target and timing','source':str(cfg.EPC_ROBUST_INCREMENTAL_SUMMARY_PATH),'interpretation':'Alternative EPC construction bounds the main result.'}]);table5.to_csv(TABLE_DIR/'Table_5_robustness_synthesis.csv',index=False)
    for name,frame in [('Table 1',table1),('Table 2',table2),('Table 3',table3),('Table 4',table4),('Table 5',table5)]:print('\n'+name);ipy_display(frame)


## Figure plan, detailed design specification, captions and scientific QA

These files freeze the scientific role of every figure independently of notebook chronology. The plan explicitly distinguishes direct Colab outputs from the one figure that may benefit from later vector-layout refinement.


In [ ]:
PLAN_ROWS = [
['Figure 1','Detailed end-to-end workflow','How do data, representations, controls, validation and evidence connect?','Frozen counts, manifest, real image assets','Vector workflow schematic','Main text','No','Descriptive assembly only','Colab vector export; optional PowerPoint/Illustrator alignment only'],
['Figure 2','Study area and actual analytical samples','Where are the final PTAL and EPC observations?','Final model table; borough boundary','Locator map + point maps','Main text','Yes','No new computation','Colab'],
['Figure 3','Outcome distributions and sampling representativeness','Did final sampling visibly change target distributions?','London-wide clean pools; final model table','Overlaid normalised histograms','Main text','No','Descriptive only','Colab'],
['Figure 4','Spatial validation design','What geography is held out by each design?','Random, borough and continuous-region folds','2×3 fold maps','Main text','Yes','No new computation','Colab'],
['Figure 5','Task-specific image support','What spatial context does each task observe?','Aerial index; Street View inventory/archive','Matched crops + thumbnail grids','Main text','Yes','Descriptive asset selection only','Colab; optional final crop alignment in Illustrator'],
['Figure 6','Primary borough-held-out benchmark','Which representation predicts best in absolute terms?','06 fold results','Mean±SD dot plot with fold points','Main text','Yes','No','Colab'],
['Figure 7','Incremental value beyond controls','What predictive information remains beyond controls?','07 paired fold deltas','Paired ΔR² dot plot','Main text','Yes','No','Colab'],
['Figure 8','Why EPC extended controls are strong','Which added EPC control drives the gain?','07 EPC control ablation','Horizontal bar chart','Main text','Yes','No','Colab'],
['Figure 9','Spatial generalisation','How does performance change under stricter spatial separation?','11 protocol summary','Categorical slope plot','Main text','Yes','No','Colab'],
['Figure 10','Observed vs OOF predictions','Do models capture extremes and improve calibration?','07 OOF predictions','Scatter + density contours','Main text','Yes','Descriptive only','Colab'],
['Figure 11','Error across target range','Where along the outcome range do errors remain?','07 OOF predictions','Quantile-group MAE and bias lines','Main text','No','Descriptive aggregation only','Colab'],
['Figure 12','Residual geography','Does residual spatial clustering remain?','11 geometric OOF predictions; Moran summary','Residual hex maps','Main text','Yes','No','Colab'],
['Figure 13','Geography of improvement','Where does full fusion reduce local-area OOF error?','07 OOF predictions; local-authority boundaries','ΔMAE choropleth','Main text','No','Descriptive aggregation only','Colab'],
['Figure 14','Model-class robustness','Do non-linear and graph heads change conclusions?','12 XGBoost; 13 GATv2 summaries','Dumbbell + ΔR² intervals','Main text','Yes','No','Colab'],
['Figure 15','EPC data and timing robustness','How sensitive is EPC evidence to target construction and record timing?','14 EPC robustness summaries','Protocol lines + year-error lines','Main text','Yes','No','Colab'],
['Appendix A1','PCA64 dimension sensitivity','Are conclusions driven by embedding dimension?','10 PCA64 summary','ΔR² interval plot','Appendix','Yes','No','Colab'],
['Appendix A2','SatCLIP added to DINOv2','Does location add beyond aerial imagery?','09 paired summary','ΔR² interval plot','Appendix','Yes','No','Colab'],
['Appendix A3','Street View coverage','Does image availability vary by local-authority area?','Final model table','Ranked area dot plot','Appendix','Yes','Descriptive only','Colab'],
]
figure_plan=pd.DataFrame(PLAN_ROWS,columns=['figure_number','title','question_answered','data_source_frozen_output','chart_type','placement','reuse_existing_figure','new_computation_needed','recommended_production_route'])
figure_plan.to_csv(TABLE_DIR/'Figure_plan_table.csv',index=False)

DESIGN_ROWS = [
['Figure 1','Single wide workflow: PTAL/EPC branches → five encoders → support → controls/model/CV → evidence','Arrows encode data flow; colours distinguish PTAL/EPC and representation roles','Counts, encoder dimensions, crop/radius rules, CV split icons','No architecture logos, performance values or decorative UMAP','The frozen pipeline links two tasks to five representations and spatially explicit evaluation.'],
['Figure 2','a London overview/inset; b PTAL points; c EPC points','Point colour=observed target','n, north arrow, scale, Thames, basemap attribution','No hexagons or explanatory text panel','The final observations cover London while retaining task-specific spatial density.'],
['Figure 3','a PTAL pool vs sample; b EPC pool vs sample','x=target; y=normalised density; fill=pool; line=sample','n and mean/median','No inferential p-value','Final sampling can be assessed directly against the clean target pools.'],
['Figure 4','Rows=tasks; columns=random/borough/continuous','Colour=categorical fold assignment','Fold key only','No ordinal colour scale or study facts panel','The three validation designs encode different generalisation assumptions.'],
['Figure 5','a 300m crop; b 150m crop; c eight PTAL street images; d four EPC street images','Frames/radii encode spatial support; image captions show distance','Centre point, exact dimensions/radii/counts','No encoder summary text panel','PTAL observes neighbourhood context; EPC observes more local context.'],
['Figure 6','a PTAL; b EPC benchmark rows','x=mean fold R²; grey=folds; colour=role','Mean R² labels','No pooled R² label','Absolute performance differs by task and representation.'],
['Figure 7','a PTAL; b EPC paired gains','x=fold ΔR²; zero=no gain','Mean±SD and numeric mean','No causal wording; zero line must remain subtle','Selected representations add predictive information beyond matched controls.'],
['Figure 8','Single EPC control ablation','x=mean paired ΔR²; bar colour=sign','Four-decimal means and SD','No representation results','Construction age explains most of the extended-control advantage.'],
['Figure 9','a PTAL; b EPC categorical slopes','x=validation design; y=pooled OOF R²; line=specification','Continuous-region value and total random→continuous change','No legend over data or paired-fold claim','Stricter spatial separation changes estimated deployment performance.'],
['Figure 10','2×2 baseline/full by task','x=observed; y=OOF prediction; contours=density','Pooled OOF R² and slope','No hexbin or intercept','Representations improve fit and reduce, but do not eliminate, regression to the mean.'],
['Figure 11','Rows=MAE/bias; columns=tasks','x=observed-target quantile group; line=model','Zero bias line and realised number of groups','No tuning or significance claim','Error concentrates at target extremes and full fusion changes its magnitude/pattern.'],
['Figure 12','2×2 baseline/full residual maps','colour=observed−predicted residual; hex=display mean','Moran’s I only','No repeated p=.005','Representations reduce residual spatial clustering without fully removing it.'],
['Figure 13','a PTAL; b EPC local-authority-area maps','colour=baseline MAE−full MAE','Positive-area count and centred colourbar','No causal local-effect wording','The predictive gain is geographically widespread or uneven, depending on task.'],
['Figure 14','Top Ridge→XGBoost; bottom GATv2−MLP','Top x=pooled R²; bottom x=paired ΔR²','Both absolute values and Δ for Ridge/XGB','No new model ranking search','Model class changes absolute accuracy but is evaluated on frozen inputs/folds.'],
['Figure 15','a EPC protocol gains; b MAE by record year','Line=model/specification; x=protocol or year','Zero line and compact legend','No mixed unrelated PCA/GAT panels','EPC conclusions are bounded by target definition and record timing.'],
]
design_spec=pd.DataFrame(DESIGN_ROWS,columns=['figure_number','panel_structure','x_y_colour_meaning','what_to_annotate','what_not_to_include','exact_intended_interpretation'])
design_spec.to_csv(TABLE_DIR/'Figure_detailed_design_spec.csv',index=False)
(MANIFEST_DIR/'Figure_detailed_design_spec.md').write_text(dataframe_to_markdown(design_spec),encoding='utf-8')

CAPTIONS = {
'Figure_1':'Figure 1. Detailed end-to-end workflow. London-wide PTAL points are reduced on a 500 m modelling grid, while EPC records are resolved and aggregated before stratified sampling across 33 local-authority areas (the 32 London boroughs and the City of London). The representation-complete sample contains 26,597 observations. Five frozen pretrained encoders, task-specific image support, matched controls, nested borough-level-held-out Ridge evaluation and robustness branches produce OOF predictions, metrics and spatial diagnostics.',
'Figure_2':'Figure 2. Study area and actual analytical samples. Greater London context and the original 6,597 PTAL and 20,000 EPC analytical locations, coloured by Access Index and mean current-efficiency score respectively. Points are not aggregated. Basemap © CARTO and OpenStreetMap contributors.',
'Figure_3':'Figure 3. Outcome distributions and sampling representativeness. Normalised target distributions for each London-wide clean pool and the final representation-complete sample. Common bins and mean/median summaries provide a descriptive check of sampling shift rather than a significance test.',
'Figure_4':'Figure 4. Spatial validation design. Fixed random, borough-level-held-out and continuous-region outer-fold assignments for PTAL and EPC. Random-fold panels show a deterministic display subset for legibility; the assignments themselves use all observations. Fold colours are categorical and have no ordinal meaning; the designs answer different deployment questions.',
'Figure_5':'Figure 5. Task-specific image support and sampling parameters. The same aerial centre at 300 m and 150 m support, together with the nearest archived Street View images under the PTAL (≤8 within 300 m) and EPC (≤4 within 150 m) rules. Examples are selected without reference to targets or model performance.',
'Figure_6':'Figure 6. Primary borough-level-held-out benchmark. Grey points are five held-out groups; coloured points and intervals are mean±SD fold R², labelled directly. Rows distinguish controls, single representations and combinations. EPC extended controls are a privileged within-EPC baseline because floor area and construction age are derived from the certificate records. These are fold means, not pooled OOF R².',
'Figure_7':'Figure 7. Incremental value beyond matched controls. Grey points are fold-level paired ΔR²; coloured points and intervals are mean±SD across five fixed borough groups. Positive values indicate added predictive information and are not causal effects.',
'Figure_8':'Figure 8. Why EPC privileged extended controls are strong. Mean paired ΔR² beyond compact EPC controls for floor area, construction age and their combination, with SD across five fixed borough-level groups. Construction age accounts for most of this within-EPC control gain; the comparison is predictive rather than causal.',
'Figure_9':'Figure 9. Spatial generalisation across evaluation designs. Pooled OOF R² for controls, DINOv2 and full combinations under random, borough-held-out and continuous-region validation. Lines link specifications for visual comparison; folds and test sets are not paired.',
'Figure_10':'Figure 10. Observed versus borough-held-out OOF predictions and calibration. Grey points are a deterministic display sample; contours use all OOF pairs. The dashed line is ideal agreement and the coloured line is a descriptive calibration fit. Pooled OOF R² and slope are held-out diagnostics.',
'Figure_11':'Figure 11. Prediction error across the target range. Borough-level-held-out OOF MAE and signed error within observed-target quantile groups for baseline and full-combination models. Because tied PTAL values are retained rather than split arbitrarily, ten requested quantiles produce nine realised PTAL groups; EPC retains ten. The descriptive aggregation shows where regression-to-the-mean and extreme-value errors remain.',
'Figure_12':"Figure 12. Residual geography under continuous-region validation. Display hexagons show mean observed-minus-predicted OOF residual; Moran's I is calculated from original residuals. Representations reduce residual spatial clustering but do not identify its cause. Permutation p-values reached the 199-permutation resolution floor, so interpretation focuses on effect size. Basemap © CARTO and OpenStreetMap contributors.",
'Figure_13':'Figure 13. Geography of representation-related improvement. Local-authority-area ΔMAE equals baseline OOF MAE minus full-model OOF MAE, so positive values indicate lower error with the full combination. The 33 areas comprise the 32 London boroughs and the City of London. This is post-hoc descriptive aggregation, not a causal local effect.',
'Figure_14':'Figure 14. Model-class robustness. Ridge and XGBoost use identical selected features and borough folds; GATv2 is compared with the matched MLP. The figure tests model-class sensitivity without changing representation inputs or initiating a new model search.',
'Figure_15':'Figure 15. EPC target, reliability and timing robustness. Representation gains across frozen EPC outcome protocols and MAE across record-year bands bound dependence on target construction and temporal composition.',
'Appendix_A1':'Appendix Figure A1. PCA64 representation-dimension sensitivity. Mean paired ΔR² for fold-local PCA-64 relative to native dimensions. This bounds dimensionality sensitivity and is not a new selection round.',
'Appendix_A2':'Appendix Figure A2. SatCLIP added to DINOv2. Mean paired ΔR² when location embedding is added to aerial embedding under frozen specifications. Near-zero change does not prove encoder equivalence.',
'Appendix_A3':'Appendix Figure A3. Street View coverage by local-authority area. Share of PTAL and EPC samples with available imagery across the 32 London boroughs and the City of London. Coverage is not image quality or predictive performance.'}
(MANIFEST_DIR/'final_captions.md').write_text('\n\n'.join(f'### {v}' for v in CAPTIONS.values()),encoding='utf-8')

figure_index=pd.DataFrame(figure_records);figure_index.to_csv(MANIFEST_DIR/'figure_index.csv',index=False)
QA_DETAILS = {
    'Figure_1':('Full frozen workflow: two tasks, five encoders, matched controls, nested CV and robustness branches','Counts and design only; no performance estimate','PTAL spatial controls; EPC compact and privileged extended controls','N/A','Methods','Two task-specific samples feed the same frozen representation-evaluation framework.','Digimap aerial and Street View provenance/permission; encoder and dataset citations.'),
    'Figure_2':('PTAL n=6,597 and EPC n=20,000 analytical locations','Observed values at individual sample locations','N/A','N/A','Methods','Both analytical samples cover all 33 Greater London local-authority areas.','CARTO/OpenStreetMap attribution; London boundary-source attribution.'),
    'Figure_3':('London-wide clean pools versus final analytical samples','Normalised density plus descriptive mean/median','N/A','N/A','Methods','Sampling shift is small for EPC and visible but bounded for PTAL.','Dataset citations only.'),
    'Figure_4':('Random, borough-level-held-out and continuous-region outer-fold assignments','Categorical assignment; no performance metric','N/A','N/A','Methods / RQ3','The three validation designs encode different deployment questions.','CARTO/OpenStreetMap attribution; boundary-source attribution.'),
    'Figure_5':('PTAL 300 m/≤8 images and EPC 150 m/≤4 images','Spatial-support dimensions and image counts','N/A','N/A','Methods','PTAL observes neighbourhood context while EPC uses more local support.','Digimap aerial licence; Street View provider attribution, licence and privacy review.'),
    'Figure_6':('Controls, individual representations and pre-specified combinations','Mean ± SD across five held-out borough-level folds; not pooled','Yes: task-matched controls shown as separate baselines','Yes','RQ1','Absolute predictive utility is task dependent and strongest for selected image/context combinations.','Model/data citations only.'),
    'Figure_7':('Selected representations/combinations paired with their matched controls','Mean paired ΔR² ± SD across five fixed folds','Yes: every ΔR² uses the declared matched control','Yes','RQ2','Representations add substantial information beyond compact controls but little beyond privileged EPC controls.','Model/data citations only.'),
    'Figure_8':('Floor area, construction age, and both added to compact EPC controls','Mean paired ΔR² ± SD across five fixed folds','Yes: compact EPC controls are the reference','Yes','RQ2','Construction age accounts for almost all of the extended-control gain.','EPC data citation only.'),
    'Figure_9':('Controls, DINOv2 and full combination under three validation designs','Pooled held-out OOF R²','Yes: PTAL spatial and EPC compact controls','Yes','RQ3','Estimated performance declines as spatial separation becomes stricter, especially for PTAL controls.','Model/data citations only.'),
    'Figure_10':('Baseline and full-combination Ridge models for PTAL and EPC','Pooled borough-level-held-out OOF R² and descriptive calibration slope','Yes: PTAL spatial and EPC compact baselines','Yes','RQ1–RQ2','Full combinations improve accuracy and reduce, but do not eliminate, regression to the mean.','Model/data citations only.'),
    'Figure_11':('Baseline and full-combination Ridge models by realised target quantile group','Pooled OOF errors aggregated descriptively within quantile groups','Yes: PTAL spatial and EPC compact baselines','Yes','RQ1–RQ2','Full combinations reduce error across most of the target range, with extremes remaining hardest.','Model/data citations only.'),
    'Figure_12':('Baseline and full-combination models under continuous-region validation','Display-hexagon mean residual; Moran’s I on individual OOF residuals','Yes: PTAL spatial and EPC compact baselines','Yes','RQ3','Representations reduce residual spatial clustering without fully removing it.','CARTO/OpenStreetMap attribution; boundary-source attribution.'),
    'Figure_13':('Baseline versus full-combination borough-level-held-out predictions aggregated to 33 areas','Local-authority-area baseline MAE minus full-model MAE','Yes: PTAL spatial and EPC compact baselines','Yes','RQ2–RQ3','Improvement is widespread but geographically uneven.','London boundary-source attribution.'),
    'Figure_14':('Ridge versus XGBoost; GATv2 versus matched MLP','Top: pooled OOF R²; bottom: mean paired ΔR² ± SD','Yes: selected control and representation specifications','Yes','Robustness','Non-linearity improves absolute accuracy, whereas GATv2 does not consistently beat the matched MLP.','Model/data citations only.'),
    'Figure_15':('Selected EPC compact/extended-control specifications across outcome and timing protocols','Panel a: mean paired ΔR²; panel b: OOF MAE within record-year bands','Yes: compact and privileged extended EPC controls','Yes','EPC robustness','The EPC representation result is stable across target definitions, while timing affects absolute error.','EPC data citation only.'),
    'Appendix_A1':('Native embedding dimensions versus fold-local PCA64','Mean paired ΔR² ± SD','Where defined by the source specification','Yes','Appendix robustness','Equalising dimension usually reduces performance and does not overturn the native benchmark.','Model/data citations only.'),
    'Appendix_A2':('DINOv2 versus DINOv2 + SatCLIP under five frozen contexts','Mean paired ΔR² ± SD','Spatial, compact and extended controls shown where applicable','Yes','Appendix RQ2','SatCLIP adds little beyond DINOv2 under most controlled specifications.','Model/data citations only.'),
    'Appendix_A3':('Street View availability for PTAL and EPC across 33 local-authority areas','Area-level share with available street-level imagery','N/A','N/A','Appendix data QA','Coverage is high but spatially uneven and is not a measure of image quality.','Street View provider/data-source citation; no imagery reproduced.'),
}
qa=[]
for _,r in figure_plan.iterrows():
    key=r.figure_number.replace(' ','_') if r.figure_number.startswith('Figure') else r.figure_number.replace('Appendix ','Appendix_')
    rec=figure_index[figure_index.figure_id.eq(key)]
    models,mean_or_pooled,controls,oof,rq,claim,licence=QA_DETAILS[key]
    qa.append({'figure_number':r.figure_number,'source_files':rec.source_files.iloc[0] if len(rec) else r.data_source_frozen_output,'source_notebook_output':r.data_source_frozen_output,'models_specifications_included':models,'metric_shown':rec.metric.iloc[0] if len(rec) else 'See design spec','mean_or_pooled':mean_or_pooled,'controls_included':controls,'oof_only':oof,'corresponding_rq':rq,'headline_claim':claim,'interpretation_boundary':rec.claim_scope.iloc[0] if len(rec) else 'Appendix descriptive scope','final_caption':CAPTIONS.get(key,''),'licence_attribution_requirement':licence})
figure_qa=pd.DataFrame(qa);figure_qa.to_csv(TABLE_DIR/'Figure_QA_table.csv',index=False);(MANIFEST_DIR/'Figure_QA_table.md').write_text(dataframe_to_markdown(figure_qa),encoding='utf-8')

from IPython.display import display as ipy_display
ipy_display(figure_plan);ipy_display(design_spec);ipy_display(figure_qa[['figure_number','metric_shown','interpretation_boundary']])


## Completion and integrity gate

The package passes only if all fifteen main figures, three appendix figures, five dissertation tables, the plan, design specification, captions, plotted-data files and QA table exist. This gate checks reporting completeness only; it never invokes model fitting.


In [ ]:
expected_main={f'Figure_{i}' for i in range(1,16)};expected_appendix={f'Appendix_A{i}' for i in range(1,4)};generated=set(figure_index.figure_id)
missing_main=sorted(expected_main-generated);missing_appendix=sorted(expected_appendix-generated);table_files=sorted(TABLE_DIR.glob('Table_[1-5]_*.csv'))
completion={'generated_at_utc':pd.Timestamp.utcnow().isoformat(),'main_figures_expected':sorted(expected_main),'main_figures_generated':sorted(generated&expected_main),'missing_main_figures':missing_main,'appendix_figures_expected':sorted(expected_appendix),'appendix_figures_generated':sorted(generated&expected_appendix),'missing_appendix_figures':missing_appendix,'tables_generated':[p.name for p in table_files],'figure_plan':str(TABLE_DIR/'Figure_plan_table.csv'),'design_spec':str(TABLE_DIR/'Figure_detailed_design_spec.csv'),'figure_qa':str(TABLE_DIR/'Figure_QA_table.csv'),'no_model_fitting_performed':True,'output_directory':str(REPORT_DIR)}
(MANIFEST_DIR/'reporting_completion.json').write_text(json.dumps(completion,indent=2),encoding='utf-8')
if missing_main: raise RuntimeError('Missing main figures: '+', '.join(missing_main))
if missing_appendix: raise RuntimeError('Missing appendix figures: '+', '.join(missing_appendix))
if len(table_files)<5: raise RuntimeError('Fewer than five dissertation tables generated.')
if len(figure_plan)!=18 or len(figure_qa)!=18: raise RuntimeError('Plan/QA does not cover 15 main + 3 appendix figures.')
print(json.dumps(completion,indent=2));print('\nStructural reporting package complete:',REPORT_DIR)
